# Steering Demo

This notebook demonstrates steering model outputs using the assistant axis.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from IPython.display import display, Markdown
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

from assistant_axis import (
    load_axis_with_metadata,
    slot_labels,
    get_config,
    ActivationSteering,
    generate_response
)

In [ ]:
# For emptying GPU memory
torch.cuda.empty_cache()

!nvidia-smi

## Load Model and Axis

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen3-32B"
MODEL_SHORT = "qwen-3-32b"
# REPO_ID = "lu-christina/assistant-axis-vectors"
AXIS_PATH = f"/workspace/{MODEL_SHORT}/{MODEL_SHORT}/roles/axis.pt"

# Get model config
config = get_config(MODEL_NAME)
TARGET_LAYER = config["target_layer"]

# Which axis slots to use for steering and capping.
# 0 = body-mean, 1..N = individual header tokens.
# Edit to combine, e.g. [0, 1] for body-mean + first header token.
STEER_SLOTS = [1]
CAP_SLOTS = [0]
CAP_THRESHOLD = 2.0  # manual threshold for slot-based capping

# Position matching mode:
#   "all"            - broadcast all vectors to all positions (Christina's original behavior)
#   "header_matched" - body vectors steer assistant response tokens only,
#                      header vectors target their matched positions only
#   "prefill_only"   - steer all positions during prefill (bakes into KV-cache), no-op during decode
#   "system_only"    - steer only the system prompt (up to its closing tag) during prefill, no-op during decode# Position matching mode:
POSITIONS_MODE = "system_only"

print(f"Model: {MODEL_NAME}")
print(f"Target layer: {TARGET_LAYER}")
print(f"Positions mode: {POSITIONS_MODE}")

In [ ]:
# Load model
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
)
print("Model loaded!")

In [ ]:
# Load axis from HuggingFace
# axis_path = hf_hub_download(repo_id=REPO_ID, filename=f"{MODEL_SHORT}/assistant_axis.pt", repo_type="dataset")
# axis_raw, axis_metadata = load_axis_with_metadata(axis_path)
# if axis_raw.ndim == 2:
#     axis_raw = axis_raw.unsqueeze(0)  # -> (1, n_layers, hidden)
# labels = slot_labels(axis_metadata)
# num_slots = axis_raw.shape[0]

# Load axis from disk
axis_raw, axis_metadata = load_axis_with_metadata(AXIS_PATH)
if axis_raw.ndim == 2:
    axis_raw = axis_raw.unsqueeze(0)  # -> (1, n_layers, hidden)
labels = slot_labels(axis_metadata)
num_slots = axis_raw.shape[0]

print(f"Axis shape: {axis_raw.shape}")
print(f"Slots ({num_slots}): {labels}")
print(f"Active steering slots: {[labels[s] for s in STEER_SLOTS]}")
print(f"Active capping slots:  {[labels[s] for s in CAP_SLOTS]}")

# Header-matched position setup
HEADER_IDS = axis_metadata.get("header_ids")
END_OF_TURN_ID = None
if HEADER_IDS is not None:
    for tok in ['<|im_end|>', '<|eot_id|>', '<end_of_turn>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != getattr(tokenizer, 'unk_token_id', None):
            END_OF_TURN_ID = tid
            break
if POSITIONS_MODE == "header_matched":
    if HEADER_IDS is None:
        print("WARNING: no header_ids in metadata — falling back to positions='all'")
    else:
        print(f"Header IDs: {dict(zip(labels[1:], HEADER_IDS))}")
        print(f"End-of-turn ID: {END_OF_TURN_ID}")

## Steering Demo

The axis points from role-playing toward default assistant behavior.
- Positive coefficient: more assistant-like
- Negative coefficient: more role-playing

In [ ]:
def _tokenize_for_position_matching(conversation):
    """Tokenize conversation for header_matched mode.
    Matches the tokenization path used by generate_response()."""
    chat_kwargs = {}
    if "qwen" in MODEL_NAME.lower():
        chat_kwargs["enable_thinking"] = False
    prompt_str = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True, **chat_kwargs)
    return tokenizer(prompt_str, return_tensors="pt").input_ids


def _position_type(slot_idx):
    """Map axis slot index to vector_position_type for header_matched mode."""
    return "body" if slot_idx == 0 else HEADER_IDS[slot_idx - 1]

def _position_kwargs(conversation, slots):
    """Build ActivationSteering kwargs for the configured POSITIONS_MODE."""
    if POSITIONS_MODE == "prefill_only":
        return dict(positions="prefill_only")
    if POSITIONS_MODE == "system_only":
        return dict(
            positions="system_only",
            input_ids=_tokenize_for_position_matching(conversation),
            end_of_turn_id=END_OF_TURN_ID,
        )
    if POSITIONS_MODE != "header_matched" or HEADER_IDS is None:
        return {}
    return dict(
        positions="header_matched",
        input_ids=_tokenize_for_position_matching(conversation),
        header_token_ids=HEADER_IDS,
        vector_position_types=[_position_type(s) for s in slots],
        end_of_turn_id=END_OF_TURN_ID,
    )

def generate_with_steering(prompt, coefficient, system_prompt=None):
    """Generate response with additive steering using STEER_SLOTS axis vectors."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    if coefficient == 0:
        response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        vectors = [axis_raw[s, TARGET_LAYER] for s in STEER_SLOTS]
        with ActivationSteering(
            model,
            steering_vectors=vectors,
            coefficients=[coefficient] * len(vectors),
            layer_indices=[TARGET_LAYER] * len(vectors),
            **_position_kwargs(conversation, STEER_SLOTS),
        ):
            response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)

    return response

In [ ]:
# Test prompt

PROMPT = "Where do you live?"
# PROMPT = "What is your name?"
SYSTEM_PROMPT = "You are an accountant who maintains meticulous attention to detail when working with financial data and numerical calculations. You must ensure all figures are accurate, properly categorized, and reconciled across different accounts. Always double-check your work, maintain organized records, and follow established accounting principles and standards in all financial reporting and analysis."

print(f"System: {SYSTEM_PROMPT}")
print(f"User: {PROMPT}")
print("=" * 60)

In [ ]:
# Generate with different steering coefficients

# POSITIONS_MODE = "all"


# 0.0 is without steering
coefficients = [0.0, -0.707, -0.841, -1.0, -1.189, -1.414, -1.682, -2.0, -2.378, -2.828, -3.364, -4.0, -4.757, -5.657, -6.727, -8.0, -9.514, -11.314, -13.454, -16.0, -19.027, -22.627, -26.909, -32.0, -38.055, -45.255, -53.817, -64.0]

# for STEER_SLOTS in [[0], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]]:
for STEER_SLOTS in [[1,2,3]]:
    for TARGET_LAYER in [24, 28, 32, 36, 40, 44, 48]:
        print(f"\nSTEER_SLOTS = {STEER_SLOTS}\nTARGET_LAYER = {TARGET_LAYER}\n")
        for coeff in coefficients:
            if coeff == 0:
                print(f"\nSTEER_SLOTS = {STEER_SLOTS}\nTARGET_LAYER = {TARGET_LAYER}\n### BASELINE")
            else:
                print(f"\n### Coefficient: {coeff}")
            print("-" * 30 + " system ---")
            
            POSITIONS_MODE = "system_only"
            response = generate_with_steering(PROMPT, coeff, SYSTEM_PROMPT)
            print(response)
            
            if len(response) > 500:
                print("...")
                
            print("-" * 30 + " prefill --")
            
            POSITIONS_MODE = "prefill_only"
            response = generate_with_steering(PROMPT, coeff, SYSTEM_PROMPT)
            print(response)
            
            if len(response) > 500:
                print("...")
                
            print("-" * 30 + " all ------")
            
            POSITIONS_MODE = "all"
            response = generate_with_steering(PROMPT, coeff, SYSTEM_PROMPT)
            print(response)
            
            if len(response) > 500:
                print("...")

## Role Transplant Steering

Instead of using the assistant axis, compute a steering vector from two raw role
vectors (Step 4 output): `role_to − role_from`.  Give the model a system prompt
for `role_from`, then steer toward `role_to`.

- **coefficient = 1.0**: nominal transplant (replace role_from behaviour with role_to)
- **coefficient < 1.0**: partial blend
- **coefficient > 1.0**: exaggerated role_to

In [ ]:
from pathlib import Path
from assistant_axis import load_role_vector

# --- Configuration ---
VECTORS_DIR = str(Path(AXIS_PATH).parent / "vectors")
ROLE_FROM = "accountant"
ROLE_TO = "pirate"
TRANSPLANT_COEFF = 1.0
# Uses STEER_SLOTS from the main config cell for slot selection

# Multi-layer: list the layers to steer at. Single-element = single-layer (no scaling).
TRANSPLANT_LAYERS = [24]  # e.g. [24, 32, 48] for multi-layer
TRANSPLANT_MULTI_LAYER = None    # "incremental", "average", or None

STEER_SLOTS = [1, 2, 3]

# --- Load vectors and compute diff ---
_vec_from, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ROLE_FROM}.pt"))
_vec_to, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ROLE_TO}.pt"))
if _vec_from.ndim == 2:
    _vec_from = _vec_from.unsqueeze(0)
if _vec_to.ndim == 2:
    _vec_to = _vec_to.unsqueeze(0)

transplant_diff = _vec_to - _vec_from
print(f"POSITIONS_MODE = {POSITIONS_MODE}")
print(f"STEER_SLOTS = {STEER_SLOTS}")
print(f"Transplant: {ROLE_FROM} → {ROLE_TO}")
print(f"Diff shape: {transplant_diff.shape}")
print(f"Layers: {TRANSPLANT_LAYERS}, multi_layer: {TRANSPLANT_MULTI_LAYER}")
for l in TRANSPLANT_LAYERS:
    print(f"  Layer {l}: "
      + ", ".join(f"{repr(labels[s])}={transplant_diff[s, l].norm():.2f}"
                  for s in STEER_SLOTS))


In [ ]:
def generate_with_transplant(prompt, coefficient=TRANSPLANT_COEFF, system_prompt=None):
    """Steer using (role_to - role_from) diff vector across TRANSPLANT_LAYERS."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    if coefficient == 0:
        return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)

    vectors = [transplant_diff[s, l] for l in TRANSPLANT_LAYERS for s in STEER_SLOTS]
    layer_indices = [l for l in TRANSPLANT_LAYERS for s in STEER_SLOTS]

    pos_kwargs = {}
    if POSITIONS_MODE == "prefill_only":
        pos_kwargs = dict(positions="prefill_only")
    elif POSITIONS_MODE == "header_matched" and HEADER_IDS is not None:
        pos_kwargs = dict(
            positions="header_matched",
            input_ids=_tokenize_for_position_matching(conversation),
            header_token_ids=HEADER_IDS,
            vector_position_types=[_position_type(s) for _ in TRANSPLANT_LAYERS for s in STEER_SLOTS],
            end_of_turn_id=END_OF_TURN_ID,
        )

    with ActivationSteering(
        model,
        steering_vectors=vectors,
        coefficients=[coefficient] * len(vectors),
        layer_indices=layer_indices,
        multi_layer=TRANSPLANT_MULTI_LAYER,
        **pos_kwargs,
    ):
        return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)


# --- Run it ---
# System prompt derived from ROLE_FROM's description field:
# "An accountant is a ..." → "You are a ..."
import json, re
with open(f"../data/roles/instructions/{ROLE_FROM}.json") as f:
    _role_desc = json.load(f)["description"]
TRANSPLANT_SYSTEM = re.sub(
    r'^An?\s+\S+\s+is\s+', 'You are ', _role_desc, count=1, flags=re.IGNORECASE)
TRANSPLANT_PROMPT = "What should I do with gold?"

print(f"System: {TRANSPLANT_SYSTEM}")
print(f"Transplant: {ROLE_FROM} → {ROLE_TO}")
print("=" * 60)

for c in [0.0, 1.0, 1.4, 2.0, 2.8, 4.0, 5.6, 8.0]:
    label = "BASELINE" if c == 0 else f"coeff={c}"
    print(f"\n### {label}")

    print("-" * 30 + " system ---")
    POSITIONS_MODE = "system_only"

    resp = generate_with_transplant(TRANSPLANT_PROMPT, c, TRANSPLANT_SYSTEM)
    print(resp[:500])
    if len(resp) > 500:
        print("...")
        
    print("-" * 30 + " prefill --")
    POSITIONS_MODE = "prefill_only"

    resp = generate_with_transplant(TRANSPLANT_PROMPT, c, TRANSPLANT_SYSTEM)
    print(resp[:500])
    if len(resp) > 500:
        print("...")
        
    print("-" * 30 + " all ------")
    POSITIONS_MODE = "all"

    resp = generate_with_transplant(TRANSPLANT_PROMPT, c, TRANSPLANT_SYSTEM)
    print(resp[:500])
    if len(resp) > 500:
        print("...")

## Header Activation Replacement

Replace header-token activations with those from a target role's stored mean
activations (Step 4 output). Instead of adding a steering direction, the model's
header activations are directly overwritten with the target role's activation pattern.

- `coefficient=1.0`: full replacement (header activations become the target role's)
- `coefficient=0.5`: 50/50 blend between original and target
- Optionally combine with body additive steering via nested context managers

In [ ]:
# --- Header Replacement Configuration ---
REPLACE_ROLE = "pirate"
REPLACE_COEFF = 1.0
REPLACE_LAYERS = [TARGET_LAYER]         # layers at which to replace header activations
REPLACE_HEADER_SLOTS = [1, 2, 3]        # header slots to replace (not body)

REPLACE_SYSTEM = """"Each time you answer a question, pick an appropriate one of the the two roles below, and answer as that person:
Role A: You are an accountant, a financial professional who manages numerical data, maintains financial records, prepares budgets, and ensures compliance with financial regulations and tax requirements.
Role B: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, values personal freedom and independence above authority, and seeks treasure and adventure through a combination of cunning strategies and bold action.
"""

# Optionally add body additive steering from the transplant diff
COMBINE_BODY_STEERING = True
BODY_STEER_COEFF = 1.0
BODY_STEER_LAYERS = [TARGET_LAYER]

# --- Load target role vector ---
_replace_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{REPLACE_ROLE}.pt"))
if _replace_vec.ndim == 2:
    _replace_vec = _replace_vec.unsqueeze(0)

print(f"Replacement role: {REPLACE_ROLE}")
print(f"Replacement shape: {_replace_vec.shape}")
print(f"Header slots: {[labels[s] for s in REPLACE_HEADER_SLOTS]}")
print(f"Layers: {REPLACE_LAYERS}")
if COMBINE_BODY_STEERING:
    print(f"+ body additive steering: coeff={BODY_STEER_COEFF}, layers={BODY_STEER_LAYERS}")

In [ ]:
def generate_with_header_replacement(prompt, system_prompt=None):
    """Replace header activations with target role's, optionally steer body additively."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    input_ids = _tokenize_for_position_matching(conversation)

    h_vectors = [_replace_vec[s, l] for l in REPLACE_LAYERS for s in REPLACE_HEADER_SLOTS]
    h_layers = [l for l in REPLACE_LAYERS for s in REPLACE_HEADER_SLOTS]
    h_pos_types = [HEADER_IDS[s - 1] for l in REPLACE_LAYERS for s in REPLACE_HEADER_SLOTS]

    header_cm = ActivationSteering(
        model,
        steering_vectors=h_vectors,
        coefficients=[REPLACE_COEFF] * len(h_vectors),
        layer_indices=h_layers,
        intervention_type="replacement",
        positions="header_matched",
        input_ids=input_ids,
        header_token_ids=HEADER_IDS,
        vector_position_types=h_pos_types,
        end_of_turn_id=END_OF_TURN_ID,
    )

    if COMBINE_BODY_STEERING:
        body_cm = ActivationSteering(
            model,
            steering_vectors=[transplant_diff[0, l] for l in BODY_STEER_LAYERS],
            coefficients=[BODY_STEER_COEFF] * len(BODY_STEER_LAYERS),
            layer_indices=list(BODY_STEER_LAYERS),
            intervention_type="addition",
            positions="header_matched",
            input_ids=input_ids,
            header_token_ids=HEADER_IDS,
            vector_position_types=["body"] * len(BODY_STEER_LAYERS),
            end_of_turn_id=END_OF_TURN_ID,
        )
        with header_cm, body_cm:
            return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        with header_cm:
            return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)


# --- Run it ---
REPLACE_PROMPT = "What should I do with gold?"

print(f"System: {REPLACE_SYSTEM}")
print(f"Header replacement: {REPLACE_ROLE} (coeff={REPLACE_COEFF})")
if COMBINE_BODY_STEERING:
    print(f"+ body steering: {ROLE_FROM}→{ROLE_TO} (coeff={BODY_STEER_COEFF})")
print("=" * 60)

print("\n### BASELINE (no intervention)")
print("-" * 40)
conv = [{"role": "system", "content": REPLACE_SYSTEM},
        {"role": "user", "content": REPLACE_PROMPT}]
baseline = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
print(baseline[:500])
if len(baseline) > 500:
    print("...")

print(f"\n### HEADER REPLACEMENT ({REPLACE_ROLE})")
print("-" * 40)
replaced = generate_with_header_replacement(REPLACE_PROMPT, TRANSPLANT_SYSTEM)
print(replaced[:500])
if len(replaced) > 500:
    print("...")

## All-Layer Header Replacement

Replace header-token activations at **every layer** with the target role's stored
mean activations. Unlike single-layer replacement, this hooks all 64 layers so the
KV-cache at layers 1+ stores K/V derived from the target role's activation pattern.
Generated tokens then attend to pirate-like header representations throughout the
entire network.

WARNING: For Gemma and Llama this will mess up the logits for generating the first
token of the agent response, so you would need to prompt around this by asking it to
say a random word then answer your question on the next line. For Qwen the four
unpatched not-thinking tokens after the header absorb this.

In [ ]:
# --- All-Layer Header Replacement Configuration ---
N_LAYERS = axis_raw.shape[1]
ALL_REPLACE_ROLE = "pirate"
ALL_REPLACE_COEFF = 1.0       # 1.0 = 100% replacement
ALL_REPLACE_SLOTS = [1, 2, 3]  # all header slots
ALL_REPLACE_LAYERS = list(range(N_LAYERS))

ALL_REPLACE_SYSTEM = """"Each time you answer a question, alternate betweeen the two roles below, Y for your first and odd-numbered replies, X for you second and even-numbered replies, and answer as that person:
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
"""

_all_replace_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ALL_REPLACE_ROLE}.pt"))
if _all_replace_vec.ndim == 2:
    _all_replace_vec = _all_replace_vec.unsqueeze(0)

print(f"All-layer header replacement: {ALL_REPLACE_ROLE}")
print(f"Layers: 0–{N_LAYERS - 1} ({N_LAYERS} hooks, {N_LAYERS * len(ALL_REPLACE_SLOTS)} vectors)")
print(f"Slots: {[labels[s] for s in ALL_REPLACE_SLOTS]}")
print(f"Coefficient: {ALL_REPLACE_COEFF}")

In [ ]:
def generate_with_all_layer_header_replacement(prompt, system_prompt=None):
    """Replace header activations at every layer with target role's stored means."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    input_ids = _tokenize_for_position_matching(conversation)

    vectors = [_all_replace_vec[s, l] for l in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    layer_indices = [l for l in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    pos_types = [HEADER_IDS[s - 1] for _ in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]

    with ActivationSteering(
        model,
        steering_vectors=vectors,
        coefficients=[ALL_REPLACE_COEFF] * len(vectors),
        layer_indices=layer_indices,
        intervention_type="replacement",
        positions="header_matched",
        input_ids=input_ids,
        header_token_ids=HEADER_IDS,
        vector_position_types=pos_types,
        end_of_turn_id=END_OF_TURN_ID,
    ):
        return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)


# --- Run it ---
ALL_REPLACE_PROMPT = "What should I do with Stocks and Shares?"

print(f"System: {ALL_REPLACE_SYSTEM}")
print(f"All-layer header replacement → {ALL_REPLACE_ROLE}")
print("=" * 60)

print("\n### BASELINE (no intervention)")
print("-" * 40)
conv = [{"role": "system", "content": ALL_REPLACE_SYSTEM},
        {"role": "user", "content": ALL_REPLACE_PROMPT}]
baseline = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
print(baseline[:500])
if len(baseline) > 500:
    print("...")

ALL_REPLACE_ROLE = "pirate"

_all_replace_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ALL_REPLACE_ROLE}.pt"))
if _all_replace_vec.ndim == 2:
    _all_replace_vec = _all_replace_vec.unsqueeze(0)

print(f"All-layer header replacement: {ALL_REPLACE_ROLE}")
print(f"Layers: 0–{N_LAYERS - 1} ({N_LAYERS} hooks, {N_LAYERS * len(ALL_REPLACE_SLOTS)} vectors)")
print(f"Slots: {[labels[s] for s in ALL_REPLACE_SLOTS]}")
print(f"Coefficient: {ALL_REPLACE_COEFF}")

print(f"\n### ALL-LAYER HEADER REPLACEMENT → {ALL_REPLACE_ROLE}")
print("-" * 40)
replaced = generate_with_all_layer_header_replacement(ALL_REPLACE_PROMPT, TRANSPLANT_SYSTEM)
print(replaced[:500])
if len(replaced) > 500:
    print("...")
    
ALL_REPLACE_ROLE = "bard"

_all_replace_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ALL_REPLACE_ROLE}.pt"))
if _all_replace_vec.ndim == 2:
    _all_replace_vec = _all_replace_vec.unsqueeze(0)

print(f"All-layer header replacement: {ALL_REPLACE_ROLE}")
print(f"Layers: 0–{N_LAYERS - 1} ({N_LAYERS} hooks, {N_LAYERS * len(ALL_REPLACE_SLOTS)} vectors)")
print(f"Slots: {[labels[s] for s in ALL_REPLACE_SLOTS]}")
print(f"Coefficient: {ALL_REPLACE_COEFF}")

print(f"\n### ALL-LAYER HEADER REPLACEMENT → {ALL_REPLACE_ROLE}")
print("-" * 40)
replaced = generate_with_all_layer_header_replacement(ALL_REPLACE_PROMPT, TRANSPLANT_SYSTEM)
print(replaced[:500])
if len(replaced) > 500:
    print("...")

In [ ]:
import torch

pirate_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "pirate.pt"))
bard_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "bard.pt"))
if pirate_vec.ndim == 2: pirate_vec = pirate_vec.unsqueeze(0)
if bard_vec.ndim == 2: bard_vec = bard_vec.unsqueeze(0)

print("=== Are pirate and bard header vectors actually different? ===\n")
print(f"{'Slot':>15s}  {'Layer':>5s}  {'cos':>8s}  {'|diff|':>10s}  {'|pirate|':>10s}  {'|bard|':>10s}")
print("-" * 65)
for s in ALL_REPLACE_SLOTS:
    for l in [0, 16, 32, 48, 63]:
        pv = pirate_vec[s, l].float()
        bv = bard_vec[s, l].float()
        cos = torch.nn.functional.cosine_similarity(pv.unsqueeze(0), bv.unsqueeze(0)).item()
        diff = (pv - bv).norm().item()
        print(f"{labels[s]:>15s}  {l:5d}  {cos:8.4f}  {diff:10.2f}  {pv.norm().item():10.2f}  {bv.norm().item():10.2f}")
    print()

print("\n=== Sanity: is the function seeing the right vectors? ===\n")

_all_replace_vec = pirate_vec
vectors_p = [_all_replace_vec[s, 32].float() for s in ALL_REPLACE_SLOTS]

_all_replace_vec = bard_vec
vectors_b = [_all_replace_vec[s, 32].float() for s in ALL_REPLACE_SLOTS]

for i, s in enumerate(ALL_REPLACE_SLOTS):
    cos = torch.nn.functional.cosine_similarity(vectors_p[i].unsqueeze(0), vectors_b[i].unsqueeze(0)).item()
    same = torch.equal(vectors_p[i].half(), vectors_b[i].half())
    print(f"Slot {s} ({labels[s]}): cos={cos:.4f}, identical={same}")

default_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "default.pt"))
if default_vec.ndim == 2: default_vec = default_vec.unsqueeze(0)
print("=== Cosine similarity of (role - default) residuals ===\n")
print(f"{'Slot':>15s}  {'Layer':>5s}  {'cos':>8s}  {'|Δpir|':>10s}  {'|Δbard|':>10s}  {'|diff|':>10s}")
print("-" * 70)
for s in ALL_REPLACE_SLOTS:
    for l in [0, 8, 16, 24, 32, 40, 48, 56, 63]:
        dp = (pirate_vec[s, l] - default_vec[s, l]).float()
        db = (bard_vec[s, l] - default_vec[s, l]).float()
        cos = torch.nn.functional.cosine_similarity(dp.unsqueeze(0), db.unsqueeze(0)).item()
        print(f"{labels[s]:>15s}  {l:5d}  {cos:8.4f}  {dp.norm().item():10.2f}  {db.norm().item():10.2f}  {(dp-db).norm().item():10.2f}")
    print()

In [ ]:
import torch
from contextlib import nullcontext

# --- Setup ---
conv = [{"role": "system", "content": ALL_REPLACE_SYSTEM},
        {"role": "user", "content": ALL_REPLACE_PROMPT}]
input_ids = _tokenize_for_position_matching(conv)
tokens = tokenizer.apply_chat_template(
    conv, return_tensors="pt", add_generation_prompt=True
).to(model.device)
layer_list = model.model.layers
LAYERS = [0, 8, 16, 24, 32, 40, 48, 56, 63]
cosf = torch.nn.functional.cosine_similarity

# --- Find last assistant header sequence ---
ids = tokens.squeeze().tolist()
hdr_pos = {}
for i in range(len(ids) - 2):
    if (ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1]
            and ids[i+2] == HEADER_IDS[2]):
        hdr_pos = {1: i, 2: i + 1, 3: i + 2}
print("Assistant header positions:",
      {repr(labels[s]): p for s, p in hdr_pos.items()})

# --- Helpers ---
def capture(steerer=None):
    out = {s: {} for s in hdr_pos}
    hooks = []
    def mk(layer_idx):
        def fn(_, __, o):
            t = o[0] if isinstance(o, tuple) else o
            for s, p in hdr_pos.items():
                out[s][layer_idx] = t[0, p].detach().cpu().float()
        return fn
    ctx = steerer if steerer is not None else nullcontext()
    with ctx:
        for l in LAYERS:
            hooks.append(layer_list[l].register_forward_hook(mk(l)))
        with torch.no_grad():
            model(tokens, use_cache=False)
        for h in hooks:
            h.remove()
    return out

def mk_steerer(vec):
    vecs = [vec[s, l] for l in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    lidx = [l for l in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    ptypes = [HEADER_IDS[s - 1] for _ in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    return ActivationSteering(
        model, steering_vectors=vecs,
        coefficients=[ALL_REPLACE_COEFF] * len(vecs),
        layer_indices=lidx, intervention_type="replacement",
        positions="header_matched", input_ids=input_ids,
        header_token_ids=HEADER_IDS, vector_position_types=ptypes,
        end_of_turn_id=END_OF_TURN_ID)

# --- 3 forward passes ---

import time

print("Capturing: unpatched...", end=" ", flush=True)
t0 = time.time()
A_none = capture()
t1 = time.time()
print(f"({t1-t0:.2f}s)  pirate-patched...", end=" ", flush=True)
A_pir = capture(mk_steerer(pirate_vec))
t2 = time.time()
print(f"({t2-t1:.2f}s)  bard-patched...", end=" ", flush=True)
A_brd = capture(mk_steerer(bard_vec))
t3 = time.time()
print(f"({t3-t2:.2f}s)  done. Total: {t3-t0:.2f}s\n")

# ==============================================================
# PART 1: Verify replacement hits the target
# ==============================================================
print("=" * 80)
print("PART 1: Replacement verification — captured vs stored target")
print("  Expect cos≈1.0, |residual|≈0 if replacement is firing")
print("=" * 80)

for tag, A, ref in [("PIRATE", A_pir, pirate_vec), ("BARD", A_brd, bard_vec)]:
    print(f"\n  {tag} replacement:")
    print(f"  {'Slot':>12s} {'Ly':>3s} {'cos(act,tgt)':>12s} {'|act−tgt|':>10s} {'|tgt|':>10s}")
    print("  " + "-" * 52)
    for s in ALL_REPLACE_SLOTS:
        for l in LAYERS:
            tgt = ref[s, l].float()
            c = cosf(A[s][l].unsqueeze(0), tgt.unsqueeze(0)).item()
            d = (A[s][l] - tgt).norm().item()
            print(f"  {repr(labels[s]):>12s} {l:3d} {c:12.6f} {d:10.4f} {tgt.norm().item():10.2f}")
        print()

# ==============================================================
# PART 2: Unpatched — which stored role is it closest to?
# ==============================================================
print("=" * 80)
print("PART 2: Unpatched residual (act − default) vs role references (role − default)")
print("=" * 80)

for s in ALL_REPLACE_SLOTS:
    print(f"\n  Slot {s} ({repr(labels[s])})")
    print(f"  {'Ly':>3} {'cos→pir':>9s} {'cos→brd':>9s} "
          f"{'‖Δ→pir‖':>9s} {'‖Δ→brd‖':>9s}  closer(cos)  closer(norm)")
    print("  " + "-" * 75)
    for l in LAYERS:
        ru = A_none[s][l] - default_vec[s, l].float()
        rp = (pirate_vec[s, l] - default_vec[s, l]).float()
        rb = (bard_vec[s, l] - default_vec[s, l]).float()
        cp = cosf(ru.unsqueeze(0), rp.unsqueeze(0)).item()
        cb = cosf(ru.unsqueeze(0), rb.unsqueeze(0)).item()
        dp = (ru - rp).norm().item()
        db = (ru - rb).norm().item()
        print(f"  {l:3d} {cp:9.4f} {cb:9.4f} {dp:9.2f} {db:9.2f}"
              f"  {'PIRATE' if cp > cb else 'bard':>11s}"
              f"  {'PIRATE' if dp < db else 'bard':>12s}")

# ==============================================================
# PART 3: All-pairs cosine & norm-of-diff  (U=unpatched, P=pirate, B=bard)
# ==============================================================
print("\n" + "=" * 100)
print("PART 3: Pairwise cosine & ‖diff‖ of residuals  (U=unpatched, P=pirate, B=bard)")
print("=" * 100)

for s in ALL_REPLACE_SLOTS:
    print(f"\n  Slot {s} ({repr(labels[s])})")
    print(f"  {'Ly':>3} {'cos(U,P)':>9s} {'cos(U,B)':>9s} {'cos(P,B)':>9s}  closest_cos"
          f"  {'‖U−P‖':>9s} {'‖U−B‖':>9s} {'‖P−B‖':>9s}  closest_norm")
    print("  " + "-" * 95)
    for l in LAYERS:
        ru = A_none[s][l] - default_vec[s, l].float()
        rp = (pirate_vec[s, l] - default_vec[s, l]).float()
        rb = (bard_vec[s, l] - default_vec[s, l]).float()
        c_up = cosf(ru.unsqueeze(0), rp.unsqueeze(0)).item()
        c_ub = cosf(ru.unsqueeze(0), rb.unsqueeze(0)).item()
        c_pb = cosf(rp.unsqueeze(0), rb.unsqueeze(0)).item()
        d_up = (ru - rp).norm().item()
        d_ub = (ru - rb).norm().item()
        d_pb = (rp - rb).norm().item()
        cos_pairs = {"U-P": c_up, "U-B": c_ub, "P-B": c_pb}
        nrm_pairs = {"U-P": d_up, "U-B": d_ub, "P-B": d_pb}
        print(f"  {l:3d} {c_up:9.4f} {c_ub:9.4f} {c_pb:9.4f}  {max(cos_pairs, key=cos_pairs.get):>11s}"
              f"  {d_up:9.2f} {d_ub:9.2f} {d_pb:9.2f}  {min(nrm_pairs, key=nrm_pairs.get):>12s}")

In [ ]:
import torch, time

ALL_LAYERS = list(range(N_LAYERS))

# Re-capture at ALL layers (needed for all-layer replacement)
def capture_all(tokens, positions):
    out = {s: {} for s in positions}
    hooks = []
    def mk(layer_idx):
        def fn(_, __, o):
            t = o[0] if isinstance(o, tuple) else o
            for s, p in positions.items():
                out[s][layer_idx] = t[0, p].detach().cpu()
        return fn
    for l in ALL_LAYERS:
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("Capturing all 64 layers: pirate-first...", end=" ", flush=True)
full_pir = capture_all(tokens_pir, hpos_pir)
t1 = time.time()
print(f"({t1-t0:.2f}s)  bard-first...", end=" ", flush=True)
full_brd = capture_all(tokens_brd, hpos_brd)
t2 = time.time()
print(f"({t2-t1:.2f}s)  done.\n")

# Build steerer from live-captured activations
def mk_live_steerer(live_acts, conv):
    input_ids = _tokenize_for_position_matching(conv)
    vecs = []
    lidx = []
    ptypes = []
    for l in ALL_LAYERS:
        for s in ALL_REPLACE_SLOTS:
            vecs.append(live_acts[s][l].to(device=model.device))
            lidx.append(l)
            ptypes.append(HEADER_IDS[s - 1])
    return ActivationSteering(
        model, steering_vectors=vecs,
        coefficients=[1.0] * len(vecs),
        layer_indices=lidx, intervention_type="replacement",
        positions="header_matched", input_ids=input_ids,
        header_token_ids=HEADER_IDS, vector_position_types=ptypes,
        end_of_turn_id=END_OF_TURN_ID)

conv_pir = [{"role": "system", "content": SYS_PIR},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]
conv_brd = [{"role": "system", "content": SYS_BRD},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]

# --- 4 generations ---
print("=" * 70)
print("1. BASELINE: Pirate-first prompt (expect pirate)")
print("-" * 70)
t0 = time.time()
r1 = generate_response(model, tokenizer, conv_pir, max_new_tokens=512, do_sample=False)
print(f"[{time.time()-t0:.1f}s]")
print(r1[:500])
if len(r1) > 500: print("...")

print("\n" + "=" * 70)
print("2. BASELINE: Bard-first prompt (expect bard)")
print("-" * 70)
t0 = time.time()
r2 = generate_response(model, tokenizer, conv_brd, max_new_tokens=512, do_sample=False)
print(f"[{time.time()-t0:.1f}s]")
print(r2[:500])
if len(r2) > 500: print("...")

print("\n" + "=" * 70)
print("3. CROSS-PATCH: Pirate-first prompt + BARD's live headers")
print("   (pirate prompt, but headers say 'bard' — does it switch?)")
print("-" * 70)
t0 = time.time()
with mk_live_steerer(full_brd, conv_pir):
    r3 = generate_response(model, tokenizer, conv_pir, max_new_tokens=512, do_sample=False)
print(f"[{time.time()-t0:.1f}s]")
print(r3[:500])
if len(r3) > 500: print("...")

print("\n" + "=" * 70)
print("4. CROSS-PATCH: Bard-first prompt + PIRATE's live headers")
print("   (bard prompt, but headers say 'pirate' — does it switch?)")
print("-" * 70)
t0 = time.time()
with mk_live_steerer(full_pir, conv_brd):
    r4 = generate_response(model, tokenizer, conv_brd, max_new_tokens=512, do_sample=False)
print(f"[{time.time()-t0:.1f}s]")
print(r4[:500])
if len(r4) > 500: print("...")

In [ ]:
import torch, time

layer_list = model.model.layers
LAYERS = [0, 8, 16, 24, 32, 40, 48, 56, 63]
cosf = torch.nn.functional.cosine_similarity

# Two prompts: only X/Y swapped in first line
SYS_PIR = """Each time you answer a question, alternate betweeen the two roles below, Y for your first and odd-numbered replies, X for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

SYS_BRD = """Each time you answer a question, alternate betweeen the two roles below, X for your first and odd-numbered replies, Y for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

def tokenize(sys_prompt):
    conv = [{"role": "system", "content": sys_prompt},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

tokens_pir = tokenize(SYS_PIR)
tokens_brd = tokenize(SYS_BRD)

# Find last assistant header in each
def find_asst_header(tokens):
    ids = tokens.squeeze().tolist()
    pos = {}
    for i in range(len(ids) - 2):
        if (ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1]
                and ids[i+2] == HEADER_IDS[2]):
            pos = {1: i, 2: i + 1, 3: i + 2}
    return pos

hpos_pir = find_asst_header(tokens_pir)
hpos_brd = find_asst_header(tokens_brd)
print(f"Pirate-first prompt: {tokens_pir.shape[1]} tokens, header at {hpos_pir}")
print(f"Bard-first prompt:   {tokens_brd.shape[1]} tokens, header at {hpos_brd}")

# Capture helper
def capture_acts(tokens, positions):
    out = {s: {} for s in positions}
    hooks = []
    def mk(layer_idx):
        def fn(_, __, o):
            t = o[0] if isinstance(o, tuple) else o
            for s, p in positions.items():
                out[s][layer_idx] = t[0, p].detach().cpu().float()
        return fn
    for l in LAYERS:
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

# --- Capture ---
t0 = time.time()
print("Capturing: pirate-first...", end=" ", flush=True)
act_pir = capture_acts(tokens_pir, hpos_pir)
t1 = time.time()
print(f"({t1-t0:.2f}s)  bard-first...", end=" ", flush=True)
act_brd = capture_acts(tokens_brd, hpos_brd)
t2 = time.time()
print(f"({t2-t1:.2f}s)  done.\n")

# ==============================================================
# Compare live diff vs stored diff
# ==============================================================
print("=" * 90)
print("Live diff (pirate_prompt − bard_prompt) vs stored diff (pirate.pt − bard.pt)")
print("  cosine: alignment of the role-contrast direction")
print("  ratio:  ‖live‖ / ‖stored‖  (how strong the live signal is vs stored)")
print("=" * 90)

for s in [1, 2, 3]:
    print(f"\n  Slot {s} ({repr(labels[s])})")
    print(f"  {'Ly':>3}  {'cosine':>8s}  {'‖live‖':>10s}  {'‖stored‖':>10s}  {'ratio':>7s}")
    print("  " + "-" * 48)
    for l in LAYERS:
        live = act_pir[s][l] - act_brd[s][l]
        stored = (pirate_vec[s, l] - bard_vec[s, l]).float()
        cos = cosf(live.unsqueeze(0), stored.unsqueeze(0)).item()
        ln = live.norm().item()
        sn = stored.norm().item()
        print(f"  {l:3d}  {cos:8.4f}  {ln:10.2f}  {sn:10.2f}  {ln/sn:7.3f}")

# ==============================================================
# Bonus: how big is the live diff vs the full activation norm?
# ==============================================================
print("\n" + "=" * 90)
print("Signal-to-total ratio: ‖live_diff‖ / mean(‖act_pir‖, ‖act_brd‖)")
print("=" * 90)

for s in [1, 2, 3]:
    print(f"\n  Slot {s} ({repr(labels[s])})")
    print(f"  {'Ly':>3}  {'‖diff‖':>10s}  {'mean‖act‖':>10s}  {'ratio%':>8s}")
    print("  " + "-" * 40)
    for l in LAYERS:
        live = act_pir[s][l] - act_brd[s][l]
        mean_norm = (act_pir[s][l].norm().item() + act_brd[s][l].norm().item()) / 2
        pct = 100 * live.norm().item() / mean_norm if mean_norm > 0 else 0
        print(f"  {l:3d}  {live.norm().item():10.2f}  {mean_norm:10.2f}  {pct:7.2f}%")

In [ ]:
import torch, time
from contextlib import nullcontext

# Verify: during cross-patched generation, are headers actually being
# replaced with the OTHER prompt's activations?

layer_list = model.model.layers
CHECK_LAYERS = [0, 16, 32, 48, 63]
cosf = torch.nn.functional.cosine_similarity

def generate_and_verify(conv, tokens, source_acts, source_label, hpos):
    """Generate with cross-patching and capture what the headers actually become."""
    input_ids = _tokenize_for_position_matching(conv)

    vecs, lidx, ptypes = [], [], []
    for l in range(N_LAYERS):
        for s in ALL_REPLACE_SLOTS:
            vecs.append(source_acts[s][l].to(device=model.device))
            lidx.append(l)
            ptypes.append(HEADER_IDS[s - 1])

    steerer = ActivationSteering(
        model, steering_vectors=vecs,
        coefficients=[1.0] * len(vecs),
        layer_indices=lidx, intervention_type="replacement",
        positions="header_matched", input_ids=input_ids,
        header_token_ids=HEADER_IDS, vector_position_types=ptypes,
        end_of_turn_id=END_OF_TURN_ID)

    captured = {s: {} for s in hpos}
    hooks = []
    call_counts = {}

    def mk(layer_idx):
        def fn(_, __, o):
            call_counts[layer_idx] = call_counts.get(layer_idx, 0) + 1
            if call_counts[layer_idx] > 1:
                return  # skip decode steps
            t = o[0] if isinstance(o, tuple) else o
            for s, p in hpos.items():
                captured[s][layer_idx] = t[0, p].detach().cpu().float()
        return fn

    with steerer:
        for l in CHECK_LAYERS:
            hooks.append(layer_list[l].register_forward_hook(mk(l)))
        try:
            with torch.no_grad():
                out = model.generate(tokens, max_new_tokens=20, do_sample=False)
        finally:
            for h in hooks:
                h.remove()

    text = tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

    print(f"\n  Replacement source: {source_label}")
    print(f"  Generated: {text[:120]}...")
    print(f"\n  {'Slot':>12s} {'Ly':>3s} {'cos(captured,source)':>20s} {'|cap−src|':>10s} {'|source|':>10s}")
    print("  " + "-" * 60)
    for s in ALL_REPLACE_SLOTS:
        for l in CHECK_LAYERS:
            src = source_acts[s][l].float()
            cap = captured[s][l]
            c = cosf(cap.unsqueeze(0), src.unsqueeze(0)).item()
            d = (cap - src).norm().item()
            print(f"  {repr(labels[s]):>12s} {l:3d} {c:20.6f} {d:10.4f} {src.norm().item():10.2f}")
        print()

    return text

# Show what we're using
print("=" * 75)
print("VERIFICATION: confirm cross-patching replaces with correct activations")
print("=" * 75)

# Also show the source activations differ between pirate/bard
print("\n--- Source activation diff (pirate vs bard) at check layers ---")
print(f"  {'Slot':>12s} {'Ly':>3s} {'cos(pir,brd)':>13s} {'|pir−brd|':>10s}")
print("  " + "-" * 42)
for s in ALL_REPLACE_SLOTS:
    for l in CHECK_LAYERS:
        pv = full_pir[s][l].float()
        bv = full_brd[s][l].float()
        c = cosf(pv.unsqueeze(0), bv.unsqueeze(0)).item()
        d = (pv - bv).norm().item()
        print(f"  {repr(labels[s]):>12s} {l:3d} {c:13.6f} {d:10.4f}")
    print()

print("\n" + "=" * 75)
print("CROSS-PATCH 1: Pirate prompt + bard's live headers")
print("=" * 75)
r3 = generate_and_verify(conv_pir, tokens_pir, full_brd, "BARD live acts", hpos_pir)

print("\n" + "=" * 75)
print("CROSS-PATCH 2: Bard prompt + pirate's live headers")
print("=" * 75)
r4 = generate_and_verify(conv_brd, tokens_brd, full_pir, "PIRATE live acts", hpos_brd)

# Final: compare to baselines
print("\n" + "=" * 75)
print("BASELINE COMPARISON")
print("=" * 75)
r1 = generate_response(model, tokenizer, conv_pir, max_new_tokens=20, do_sample=False)
r2 = generate_response(model, tokenizer, conv_brd, max_new_tokens=20, do_sample=False)
print(f"  Baseline pirate:     {r1[:120]}...")
print(f"  Cross-patch pir+brd: {r3[:120]}...")
print(f"  Match: {r1[:120] == r3[:120]}")
print()
print(f"  Baseline bard:       {r2[:120]}...")
print(f"  Cross-patch brd+pir: {r4[:120]}...")
print(f"  Match: {r2[:120] == r4[:120]}")

In [ ]:
# Remove stale hooks from all layers
for l in range(N_LAYERS):
    layer_list[l]._forward_hooks.clear()
print(f"Cleared hooks from {N_LAYERS} layers")


In [ ]:
import torch, time

layer_list = model.model.layers
CHECK_LAYERS = [0, 16, 32, 48, 63]
ALL_LAYERS = list(range(N_LAYERS))
cosf = torch.nn.functional.cosine_similarity

# --- Prompts ---
SYS_PIR = """Each time you answer a question, alternate betweeen the two roles below, Y for your first and odd-numbered replies, X for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

SYS_BRD = """Each time you answer a question, alternate betweeen the two roles below, X for your first and odd-numbered replies, Y for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

conv_pir = [{"role": "system", "content": SYS_PIR},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]
conv_brd = [{"role": "system", "content": SYS_BRD},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]

# Tokenize with thinking disabled (matching generate_response behavior)
def tokenize_no_think(conv):
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True,
        enable_thinking=False
    ).to(model.device)

tokens_pir = tokenize_no_think(conv_pir)
tokens_brd = tokenize_no_think(conv_brd)

# Find last assistant header
def find_asst_header(tokens):
    ids = tokens.squeeze().tolist()
    pos = {}
    for i in range(len(ids) - 2):
        if (ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1]
                and ids[i+2] == HEADER_IDS[2]):
            pos = {1: i, 2: i + 1, 3: i + 2}
    return pos

hpos_pir = find_asst_header(tokens_pir)
hpos_brd = find_asst_header(tokens_brd)
print(f"Pirate prompt: {tokens_pir.shape[1]} tokens, header at {hpos_pir}")
print(f"Bard prompt:   {tokens_brd.shape[1]} tokens, header at {hpos_brd}")
print(f"Thinking: disabled (enable_thinking=False)")

# --- Capture all layers (no steering) ---
def capture_all(tokens, positions):
    out = {s: {} for s in positions}
    hooks = []
    def mk(layer_idx):
        def fn(_, __, o):
            t = o[0] if isinstance(o, tuple) else o
            for s, p in positions.items():
                out[s][layer_idx] = t[0, p].detach().cpu()
        return fn
    for l in ALL_LAYERS:
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("Capturing all 64 layers: pirate-first...", end=" ", flush=True)
full_pir = capture_all(tokens_pir, hpos_pir)
t1 = time.time()
print(f"({t1-t0:.2f}s)  bard-first...", end=" ", flush=True)
full_brd = capture_all(tokens_brd, hpos_brd)
t2 = time.time()
print(f"({t2-t1:.2f}s)  done.\n")

# --- Build steerer from live activations ---
def mk_live_steerer(live_acts, conv):
    input_ids = _tokenize_for_position_matching(conv)
    vecs, lidx, ptypes = [], [], []
    for l in ALL_LAYERS:
        for s in ALL_REPLACE_SLOTS:
            vecs.append(live_acts[s][l].to(device=model.device))
            lidx.append(l)
            ptypes.append(HEADER_IDS[s - 1])
    return ActivationSteering(
        model, steering_vectors=vecs,
        coefficients=[1.0] * len(vecs),
        layer_indices=lidx, intervention_type="replacement",
        positions="header_matched", input_ids=input_ids,
        header_token_ids=HEADER_IDS, vector_position_types=ptypes,
        end_of_turn_id=END_OF_TURN_ID)

# --- Generate + verify ---
def generate_and_verify(conv, tokens, source_acts, source_label, hpos):
    input_ids = _tokenize_for_position_matching(conv)
    vecs, lidx, ptypes = [], [], []
    for l in ALL_LAYERS:
        for s in ALL_REPLACE_SLOTS:
            vecs.append(source_acts[s][l].to(device=model.device))
            lidx.append(l)
            ptypes.append(HEADER_IDS[s - 1])

    steerer = ActivationSteering(
        model, steering_vectors=vecs,
        coefficients=[1.0] * len(vecs),
        layer_indices=lidx, intervention_type="replacement",
        positions="header_matched", input_ids=input_ids,
        header_token_ids=HEADER_IDS, vector_position_types=ptypes,
        end_of_turn_id=END_OF_TURN_ID)

    captured = {s: {} for s in hpos}
    hooks = []
    call_counts = {}

    def mk(layer_idx):
        def fn(_, __, o):
            call_counts[layer_idx] = call_counts.get(layer_idx, 0) + 1
            if call_counts[layer_idx] > 1:
                return
            t = o[0] if isinstance(o, tuple) else o
            for s, p in hpos.items():
                captured[s][layer_idx] = t[0, p].detach().cpu().float()
        return fn

    with steerer:
        for l in CHECK_LAYERS:
            hooks.append(layer_list[l].register_forward_hook(mk(l)))
        try:
            with torch.no_grad():
                out = model.generate(tokens, max_new_tokens=512, do_sample=False,
                                     pad_token_id=tokenizer.pad_token_id)
        finally:
            for h in hooks:
                h.remove()

    text = tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

    print(f"\n  Replacement source: {source_label}")
    print(f"  Output (first 300 chars):\n    {text[:300]}")
    if len(text) > 300: print("    ...")
    print(f"\n  {'Slot':>12s} {'Ly':>3s} {'cos(captured,source)':>20s} {'|cap−src|':>10s}")
    print("  " + "-" * 50)
    for s in ALL_REPLACE_SLOTS:
        for l in CHECK_LAYERS:
            src = source_acts[s][l].float()
            cap = captured[s][l]
            c = cosf(cap.unsqueeze(0), src.unsqueeze(0)).item()
            d = (cap - src).norm().item()
            print(f"  {repr(labels[s]):>12s} {l:3d} {c:20.6f} {d:10.4f}")
        print()
    return text

# --- Source diff sanity check ---
print("=" * 80)
print("SOURCE DIFF: pirate vs bard live activations at header positions")
print("=" * 80)
print(f"  {'Slot':>12s} {'Ly':>3s} {'cos(pir,brd)':>13s} {'|pir−brd|':>10s}")
print("  " + "-" * 42)
for s in ALL_REPLACE_SLOTS:
    for l in CHECK_LAYERS:
        pv = full_pir[s][l].float()
        bv = full_brd[s][l].float()
        c = cosf(pv.unsqueeze(0), bv.unsqueeze(0)).item()
        d = (pv - bv).norm().item()
        print(f"  {repr(labels[s]):>12s} {l:3d} {c:13.6f} {d:10.4f}")
    print()

# --- Baselines ---
print("=" * 80)
print("BASELINES (generate_response, thinking disabled)")
print("=" * 80)
t0 = time.time()
r1 = generate_response(model, tokenizer, conv_pir, max_new_tokens=512, do_sample=False)
t1 = time.time()
print(f"\n  Pirate-first ({t1-t0:.1f}s):\n    {r1[:300]}")
if len(r1) > 300: print("    ...")

t0 = time.time()
r2 = generate_response(model, tokenizer, conv_brd, max_new_tokens=512, do_sample=False)
t1 = time.time()
print(f"\n  Bard-first ({t1-t0:.1f}s):\n    {r2[:300]}")
if len(r2) > 300: print("    ...")

# --- Cross-patches ---
print("\n" + "=" * 80)
print("CROSS-PATCH 1: Pirate prompt + BARD's live headers (thinking disabled)")
print("=" * 80)
r3 = generate_and_verify(conv_pir, tokens_pir, full_brd, "BARD live acts", hpos_pir)

print("\n" + "=" * 80)
print("CROSS-PATCH 2: Bard prompt + PIRATE's live headers (thinking disabled)")
print("=" * 80)
r4 = generate_and_verify(conv_brd, tokens_brd, full_pir, "PIRATE live acts", hpos_brd)

# --- Final comparison ---
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
match_3 = r1[:200] == r3[:200]
match_4 = r2[:200] == r4[:200]
print(f"  Pirate baseline vs cross-patch (first 200 chars match): {match_3}")
print(f"  Bard baseline vs cross-patch (first 200 chars match):   {match_4}")
if match_3 and match_4:
    print("  → Headers are NOT causal for role selection")
elif not match_3 and not match_4:
    print("  → Headers HAVE causal effect — inspect outputs above")
else:
    print("  → Mixed result — inspect outputs above")

In [ ]:
import torch, time

layer_list = model.model.layers

# Tokenize with thinking disabled
def tokenize_no_think(conv):
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True,
        enable_thinking=False
    ).to(model.device)

tokens_pir = tokenize_no_think(conv_pir)
tokens_brd = tokenize_no_think(conv_brd)
print(f"Pirate prompt: {tokens_pir.shape[1]} tokens")
print(f"Bard prompt:   {tokens_brd.shape[1]} tokens")

# Capture FULL layer outputs (all positions, all layers)
def capture_full(tokens):
    out = {}
    hooks = []
    for l in range(N_LAYERS):
        def mk(li):
            def fn(_, __, output):
                t = output[0] if isinstance(output, tuple) else output
                out[li] = t[0].detach().cpu()
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("Capturing full prefill: pirate...", end=" ", flush=True)
full_pir = capture_full(tokens_pir)
t1 = time.time()
print(f"({t1-t0:.2f}s)  bard...", end=" ", flush=True)
full_brd = capture_full(tokens_brd)
t2 = time.time()
mem = sum(v.numel() * v.element_size() for v in full_pir.values())
print(f"({t2-t1:.2f}s)  done. ({mem/1e6:.0f} MB per capture)\n")

# Generate with full-prefill replacement
def generate_full_replace(tokens, target_acts, max_new_tokens=512):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None  # don't touch decode
                t = output[0] if isinstance(output, tuple) else output
                repl = target_acts[li].unsqueeze(0).to(
                    device=t.device, dtype=t.dtype)
                if isinstance(output, tuple):
                    return (repl,) + output[1:]
                return repl
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Baselines ---
print("=" * 70)
print("BASELINES")
print("=" * 70)
t0 = time.time()
r1 = generate_response(model, tokenizer, conv_pir, max_new_tokens=512, do_sample=False)
print(f"\n  Pirate ({time.time()-t0:.1f}s):\n    {r1[:300]}")
if len(r1) > 300: print("    ...")

t0 = time.time()
r2 = generate_response(model, tokenizer, conv_brd, max_new_tokens=512, do_sample=False)
print(f"\n  Bard ({time.time()-t0:.1f}s):\n    {r2[:300]}")
if len(r2) > 300: print("    ...")

# --- Full cross-patches ---
print("\n" + "=" * 70)
print("FULL REPLACE: Pirate prompt → ALL positions from bard")
print("  (should produce bard-like output)")
print("=" * 70)
t0 = time.time()
r3 = generate_full_replace(tokens_pir, full_brd)
print(f"\n  ({time.time()-t0:.1f}s):\n    {r3[:300]}")
if len(r3) > 300: print("    ...")

print("\n" + "=" * 70)
print("FULL REPLACE: Bard prompt → ALL positions from pirate")
print("  (should produce pirate-like output)")
print("=" * 70)
t0 = time.time()
r4 = generate_full_replace(tokens_brd, full_pir)
print(f"\n  ({time.time()-t0:.1f}s):\n    {r4[:300]}")
if len(r4) > 300: print("    ...")

# --- Summary ---
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"  Pirate baseline:      {r1[:80]}...")
print(f"  Bard baseline:        {r2[:80]}...")
print(f"  Pirate+bard_all:      {r3[:80]}...")
print(f"  Bard+pirate_all:      {r4[:80]}...")
print()
# Check if roles swapped
pir_has_pirate = any(w in r1[:100].lower() for w in ["arrr", "matey", "pirate", "ye be"])
r3_has_bard = any(w in r3[:100].lower() for w in ["bard", "tale", "song", "story", "realm"])
r3_has_pirate = any(w in r3[:100].lower() for w in ["arrr", "matey", "pirate", "ye be"])
r4_has_pirate = any(w in r4[:100].lower() for w in ["arrr", "matey", "pirate", "ye be"])
r4_has_bard = any(w in r4[:100].lower() for w in ["bard", "tale", "song", "story", "realm"])
print(f"  Pirate+bard_all → bard-like: {r3_has_bard}, pirate-like: {r3_has_pirate}")
print(f"  Bard+pirate_all → pirate-like: {r4_has_pirate}, bard-like: {r4_has_bard}")

In [ ]:
# System-only replacement: patch positions 0 to patch_start-1, leave rest intact

print(f"System-only: patching positions 0–{patch_start-1} ({patch_start} of {len(ids)} tokens)")
print(f"Leaving post-system unpatched: positions {patch_start}–{len(ids)-1}\n")

def generate_system_replace(tokens, target_acts, end_pos, max_new_tokens=512):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = target_acts[li].to(device=t.device, dtype=t.dtype)
                modified[0, :end_pos] = src[:end_pos]
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Run ---
print("=" * 70)
print("SYSTEM-ONLY REPLACE: Pirate prompt + bard's system activations")
print("=" * 70)
t0 = time.time()
r7 = generate_system_replace(tokens_pir, full_brd, patch_start)
print(f"\n  ({time.time()-t0:.1f}s):\n    {r7[:300]}")
if len(r7) > 300: print("    ...")

print("\n" + "=" * 70)
print("SYSTEM-ONLY REPLACE: Bard prompt + pirate's system activations")
print("=" * 70)
t0 = time.time()
r8 = generate_system_replace(tokens_brd, full_pir, patch_start)
print(f"\n  ({time.time()-t0:.1f}s):\n    {r8[:300]}")
if len(r8) > 300: print("    ...")

# --- Full comparison ---
print("\n" + "=" * 70)
print("FULL COMPARISON")
print("=" * 70)
print(f"  {'Condition':<30s}  First 80 chars")
print("  " + "-" * 115)
print(f"  {'Pirate baseline':<30s}  {r1[:80]}...")
print(f"  {'Bard baseline':<30s}  {r2[:80]}...")
print(f"  {'Pir+brd headers only':<30s}  {r1[:80]}...")  # from earlier: no change
print(f"  {'Brd+pir headers only':<30s}  {r2[:80]}...")  # from earlier: no change
print(f"  {'Pir+brd post-system':<30s}  {r5[:80]}...")
print(f"  {'Brd+pir post-system':<30s}  {r6[:80]}...")
print(f"  {'Pir+brd system-only':<30s}  {r7[:80]}...")
print(f"  {'Brd+pir system-only':<30s}  {r8[:80]}...")
print(f"  {'Pir+brd ALL':<30s}  {r3[:80]}...")
print(f"  {'Brd+pir ALL':<30s}  {r4[:80]}...")

In [ ]:
import torch, time

layer_list = model.model.layers

# --- Prompts ---
SYS_PIR = """Each time you answer a question, alternate betweeen the two roles below, Y for your first and odd-numbered replies, X for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

SYS_BRD = """Each time you answer a question, alternate betweeen the two roles below, X for your first and odd-numbered replies, Y for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

conv_pir = [{"role": "system", "content": SYS_PIR},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]
conv_brd = [{"role": "system", "content": SYS_BRD},
            {"role": "user", "content": ALL_REPLACE_PROMPT}]

def tokenize_no_think(conv):
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True,
        enable_thinking=False).to(model.device)

tokens_pir = tokenize_no_think(conv_pir)
tokens_brd = tokenize_no_think(conv_brd)

# --- Find system prompt boundary ---
ids = tokens_pir.squeeze().tolist()
sys_end = ids.index(END_OF_TURN_ID)
post_sys_start = sys_end + 2  # skip <|im_end|> \n

# Find assistant header
hdr_pos = {}
for i in range(len(ids) - 2):
    if ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1] and ids[i+2] == HEADER_IDS[2]:
        hdr_pos = {1: i, 2: i + 1, 3: i + 2}
hdr_positions = sorted(hdr_pos.values())

print(f"Prompt: {len(ids)} tokens")
print(f"  System prompt:    positions 0–{post_sys_start-1} ({post_sys_start} tokens)")
print(f"  Post-system:      positions {post_sys_start}–{len(ids)-1} ({len(ids)-post_sys_start} tokens)")
print(f"  Asst header only: positions {hdr_positions} (3 tokens)")
print(f"  Thinking: disabled\n")

# --- Capture full prefill ---
def capture_full(tokens):
    out = {}
    hooks = []
    for l in range(N_LAYERS):
        def mk(li):
            def fn(_, __, output):
                t = output[0] if isinstance(output, tuple) else output
                out[li] = t[0].detach().cpu()
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("Capturing full prefill: pirate...", end=" ", flush=True)
full_pir = capture_full(tokens_pir)
t1 = time.time()
print(f"({t1-t0:.2f}s)  bard...", end=" ", flush=True)
full_brd = capture_full(tokens_brd)
t2 = time.time()
print(f"({t2-t1:.2f}s)  done.\n")

# --- Generic partial-replace generator ---
def generate_with_replace(tokens, target_acts, positions_to_patch, max_new_tokens=512):
    """Replace specific positions during prefill, leave rest intact."""
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = target_acts[li].to(device=t.device, dtype=t.dtype)
                for p in positions_to_patch:
                    modified[0, p] = src[p]
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Define the 4 regions ---
all_pos = list(range(len(ids)))
sys_pos = list(range(post_sys_start))
post_sys_pos = list(range(post_sys_start, len(ids)))

experiments = [
    ("Headers only (3 tok)",    hdr_positions),
    ("Post-system (21 tok)",    post_sys_pos),
    ("System-only (135 tok)",   sys_pos),
    ("Everything (156 tok)",    all_pos),
]

# --- Baselines ---
print("=" * 90)
print("BASELINES")
print("=" * 90)
t0 = time.time()
r_pir_base = generate_response(model, tokenizer, conv_pir, max_new_tokens=512, do_sample=False)
t1 = time.time()
r_brd_base = generate_response(model, tokenizer, conv_brd, max_new_tokens=512, do_sample=False)
t2 = time.time()
print(f"  Pirate ({t1-t0:.1f}s): {r_pir_base[:100]}...")
print(f"  Bard   ({t2-t1:.1f}s): {r_brd_base[:100]}...")

# --- Run all 8 cross-patches ---
results = {}
for label, positions in experiments:
    print(f"\n{'=' * 90}")
    print(f"REGION: {label}")
    print(f"{'=' * 90}")

    t0 = time.time()
    r_p2b = generate_with_replace(tokens_pir, full_brd, positions)
    t1 = time.time()
    r_b2p = generate_with_replace(tokens_brd, full_pir, positions)
    t2 = time.time()

    results[label] = (r_p2b, r_b2p)
    print(f"  Pir+brd ({t1-t0:.1f}s): {r_p2b[:100]}...")
    print(f"  Brd+pir ({t2-t1:.1f}s): {r_b2p[:100]}...")

# --- Summary table ---
print(f"\n{'=' * 90}")
print("SUMMARY TABLE")
print(f"{'=' * 90}")
print(f"  {'Region':<25s}  {'Pir prompt + brd acts':<30s}  {'Brd prompt + pir acts':<30s}")
print("  " + "-" * 87)
print(f"  {'Baseline':<25s}  {r_pir_base[:28]:<30s}  {r_brd_base[:28]:<30s}")
for label, (r_p2b, r_b2p) in results.items():
    print(f"  {label:<25s}  {r_p2b[:28]:<30s}  {r_b2p[:28]:<30s}")
    

In [ ]:
import torch, time

# --- Find user prompt boundary ---
ids = tokens_pir.squeeze().tolist()
# System ends at first <|im_end|>
sys_eot = ids.index(END_OF_TURN_ID)
sys_end = sys_eot + 2  # include <|im_end|> \n

# User turn ends at second <|im_end|>
user_eot = ids.index(END_OF_TURN_ID, sys_eot + 1)
# user_end = user_eot + 1  # include <|im_end|> only
user_end = user_eot + 2  # include <|im_end|> and \n

print(f"Prompt: {len(ids)} tokens")
print(f"  System prompt:     0–{sys_end-1} ({sys_end} tokens)")
print(f"  User turn:         {sys_end}–{user_end-1} ({user_end - sys_end} tokens)")
print(f"  Asst header:       {user_end}–{len(ids)-1} ({len(ids) - user_end} tokens)")
print()

# Show boundary tokens
print("Boundary tokens:")
for i in range(max(0, user_eot - 2), min(len(ids), user_end + 4)):
    marker = ""
    if i == user_end: marker = " ← asst header starts here"
    print(f"  [{i}] {ids[i]} = {repr(tokenizer.decode([ids[i]]))}{marker}")

# --- Position sets ---
sys_and_user = list(range(user_end))           # system + user
asst_header = list(range(user_end, len(ids)))   # assistant header only
all_pos = list(range(len(ids)))

experiments = [
    ("Asst header only",    asst_header,   f"{len(asst_header)} tok"),
    ("System+user",         sys_and_user,  f"{len(sys_and_user)} tok"),
    ("Everything",          all_pos,       f"{len(all_pos)} tok"),
]

# --- Baselines ---
print(f"\n{'=' * 80}")
print("BASELINES")
print("=" * 80)
t0 = time.time()
r_pir = generate_response(model, tokenizer, conv_pir, max_new_tokens=512, do_sample=False)
t1 = time.time()
r_brd = generate_response(model, tokenizer, conv_brd, max_new_tokens=512, do_sample=False)
t2 = time.time()
print(f"\n  Pirate baseline ({t1-t0:.1f}s):")
print(f"    {r_pir[:200]}")
if len(r_pir) > 200: print("    ...")
print(f"\n  Bard baseline ({t2-t1:.1f}s):")
print(f"    {r_brd[:200]}")
if len(r_brd) > 200: print("    ...")

# --- Run experiments ---
results = {}
for label, positions, tok_label in experiments:
    print(f"\n{'=' * 80}")
    print(f"REGION: {label} ({tok_label})")
    print("=" * 80)

    t0 = time.time()
    r_p2b = generate_with_replace(tokens_pir, full_brd, positions)
    t1 = time.time()
    r_b2p = generate_with_replace(tokens_brd, full_pir, positions)
    t2 = time.time()

    results[label] = (r_p2b, r_b2p)

    print(f"\n  Pir prompt + brd acts ({t1-t0:.1f}s):")
    print(f"    {r_p2b[:200]}")
    if len(r_p2b) > 200: print("    ...")
    print(f"\n  Brd prompt + pir acts ({t2-t1:.1f}s):")
    print(f"    {r_b2p[:200]}")
    if len(r_b2p) > 200: print("    ...")

# --- Summary ---
print(f"\n{'=' * 80}")
print("SUMMARY")
print("=" * 80)

for label in ["Baseline"] + [e[0] for e in experiments]:
    if label == "Baseline":
        p, b = r_pir, r_brd
    else:
        p, b = results[label]

    # Detect role
    def role_tag(text):
        t = text[:80].lower()
        if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
            return "PIRATE"
        if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
            return "BARD"
        return "???"

    print(f"\n  {label}:")
    print(f"    Pir prompt + brd acts → {role_tag(p):>6s}  |  {p[:60]}...")
    print(f"    Brd prompt + pir acts → {role_tag(b):>6s}  |  {b[:60]}...")

In [ ]:
import torch, time

# Two prompts: only X/Y swapped in first line
SYS_PIR = """Each time you answer a question, alternate betweeen the two roles below, Y for your first and odd-numbered replies, X for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

SYS_BRD = """Each time you answer a question, alternate betweeen the two roles below, X for your first and odd-numbered replies, Y for you second and even-numbered replies, and answer as that person. Start your response with the word "Response:". DO NOT first say X, Y, or which role you are resonding as.
Role X: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative
Role Y: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action."""

asst_pos = list(range(149, 156))  # 7-token asst header
all_but_asst = list(range(149))    # everything except asst header (0-148)

# Build extrapolated activation dicts
def make_extrapolated(base, other):
    """2*base - other: extrapolate PAST base, away from other."""
    result = {}
    for l in range(N_LAYERS):
        result[l] = (2 * base[l].float() - other[l].float()).to(base[l].dtype)
    return result

extra_pir = make_extrapolated(full_pir, full_brd)  # extra pirate
extra_brd = make_extrapolated(full_brd, full_pir)  # extra bard

print(f"Asst header positions: {asst_pos} (7 tokens)")
print(f"All-but-asst positions: 0–{all_but_asst[-1]} ({len(all_but_asst)} tokens)\n")

# --- Experiments ---
experiments = [
    # (label, prompt_label, tokens, target_acts, positions)
    # Pirate prompt conditions
    ("PIR baseline",           "pir", tokens_pir, None,      []),
    ("PIR + brd@header",       "pir", tokens_pir, full_brd,  asst_pos),
    ("PIR + extra_brd@header", "pir", tokens_pir, extra_brd, asst_pos),
    ("PIR + brd@all_but",      "pir", tokens_pir, full_brd,  all_but_asst),
    # Bard prompt conditions
    ("BRD baseline",           "brd", tokens_brd, None,      []),
    ("BRD + pir@header",       "brd", tokens_brd, full_pir,  asst_pos),
    ("BRD + extra_pir@header", "brd", tokens_brd, extra_pir, asst_pos),
    ("BRD + pir@all_but",      "brd", tokens_brd, full_pir,  all_but_asst),
]

results = {}
for label, prompt_label, tokens, target, positions in experiments:
    print(f"{'=' * 80}")
    print(f"{label}")
    print(f"{'=' * 80}")

    t0 = time.time()
    if target is None:
        conv = conv_pir if prompt_label == "pir" else conv_brd
        text = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
    else:
        text = generate_with_replace(tokens, target, positions)
    elapsed = time.time() - t0

    results[label] = text
    print(f"  ({elapsed:.1f}s):")
    print(f"  {text[:500]}")
    if len(text) > 500: print("  ...")
# --- Summary ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIRATE"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BARD"
    return "???"

print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\n  PIRATE PROMPT:")
for label in [e[0] for e in experiments[:4]]:
    r = results[label]
    print(f"    {label:<25s} → {role_tag(r):>6s}  | {r[:55]}...")

print(f"\n  BARD PROMPT:")
for label in [e[0] for e in experiments[4:]]:
    r = results[label]
    print(f"    {label:<25s} → {role_tag(r):>6s}  | {r[:55]}...")

In [ ]:
import torch, time

asst_pos = list(range(149, 156))
all_but_asst = list(range(149))

# Extrapolated activations
def make_extrap(base, other, scale):
    """scale*base - (scale-1)*other: extrapolate past base away from other."""
    result = {}
    for l in range(N_LAYERS):
        b, o = base[l].float(), other[l].float()
        result[l] = (scale * b - (scale - 1) * o).to(base[l].dtype)
    return result

extra_pir_2x = make_extrap(full_pir, full_brd, 2)   # 2*pir - 1*brd
extra_brd_2x = make_extrap(full_brd, full_pir, 2)   # 2*brd - 1*pir
extra_pir_4x = make_extrap(full_pir, full_brd, 4)   # 4*pir - 3*brd
extra_brd_4x = make_extrap(full_brd, full_pir, 4)   # 4*brd - 3*pir

print(f"Asst header: positions {asst_pos} (7 tokens)")
print(f"All-but-asst: 0–148 ({len(all_but_asst)} tokens)\n")

experiments = [
    ("PIR baseline",            "pir", tokens_pir, None,          []),
    ("PIR + brd@header",        "pir", tokens_pir, full_brd,      asst_pos),
    ("PIR + 2x_brd@header",    "pir", tokens_pir, extra_brd_2x,  asst_pos),
    ("PIR + 4x_brd@header",    "pir", tokens_pir, extra_brd_4x,  asst_pos),
    ("PIR + brd@all_but",       "pir", tokens_pir, full_brd,      all_but_asst),

    ("BRD baseline",            "brd", tokens_brd, None,          []),
    ("BRD + pir@header",        "brd", tokens_brd, full_pir,      asst_pos),
    ("BRD + 2x_pir@header",    "brd", tokens_brd, extra_pir_2x,  asst_pos),
    ("BRD + 4x_pir@header",    "brd", tokens_brd, extra_pir_4x,  asst_pos),
    ("BRD + pir@all_but",       "brd", tokens_brd, full_pir,      all_but_asst),
]

results = {}
for label, prompt_label, tokens, target, positions in experiments:
    print(f"{'=' * 80}")
    print(f"{label}")
    print(f"{'=' * 80}")

    t0 = time.time()
    if target is None:
        conv = conv_pir if prompt_label == "pir" else conv_brd
        text = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
    else:
        text = generate_with_replace(tokens, target, positions)
    elapsed = time.time() - t0

    results[label] = text
    print(f"  ({elapsed:.1f}s):")
    print(f"  {text[:500]}")
    if len(text) > 500: print("  ...")
    print()

# --- Summary ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIRATE"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BARD"
    return "???"

print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\n  PIRATE PROMPT:")
for label in [e[0] for e in experiments[:5]]:
    r = results[label]
    print(f"    {label:<25s} → {role_tag(r):>6s}  | {r[:55]}...")

print(f"\n  BARD PROMPT:")
for label in [e[0] for e in experiments[5:]]:
    r = results[label]
    print(f"    {label:<25s} → {role_tag(r):>6s}  | {r[:55]}...")

In [ ]:
NO, BELOW THIS

In [ ]:
import torch, time

layer_list = model.model.layers

# --- Two system prompts (only X/Y ordering differs) ---
SYS_PIR = """Each time you answer a question, answer in one of the two roles below. Use role X to reply to the first question, role Y to answer the second question. START your response with the word "Role". THEN say X, Y, or which role you are resonding as.
Role X: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
Role Y: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
"""

SYS_BRD = """Each time you answer a question, answer in one of the two roles below. Use role Y to reply to the first question, role X to answer the second question. START your response with the word "Role". THEN say Y, X, or which role you are resonding as.
Role X: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
Role Y: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
"""

PROMPT = "What should I do with Stocks and Shares?"

conv_pir = [{"role": "system", "content": SYS_PIR}, {"role": "user", "content": PROMPT}]
conv_brd = [{"role": "system", "content": SYS_BRD}, {"role": "user", "content": PROMPT}]

def tokenize_no_think(conv):
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True,
        enable_thinking=False).to(model.device)

tokens_pir = tokenize_no_think(conv_pir)
tokens_brd = tokenize_no_think(conv_brd)

# --- Find assistant header positions ---
ids = tokens_pir.squeeze().tolist()
hdr_start = None
for i in range(len(ids) - 2):
    if ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1] and ids[i+2] == HEADER_IDS[2]:
        hdr_start = i
asst_pos = list(range(hdr_start, len(ids)))
all_but_asst = list(range(hdr_start))

print(f"Prompt: {len(ids)} tokens, header starts at {hdr_start}")
print(f"Asst header: positions {asst_pos} ({len(asst_pos)} tokens)")
print(f"All-but-asst: 0–{hdr_start-1} ({len(all_but_asst)} tokens)")
for p in asst_pos:
    print(f"  [{p}] {ids[p]} = {repr(tokenizer.decode([ids[p]]))}")

# --- Capture full prefill ---
def capture_full(tokens):
    out = {}
    hooks = []
    for l in range(N_LAYERS):
        def mk(li):
            def fn(_, __, output):
                t = output[0] if isinstance(output, tuple) else output
                out[li] = t[0].detach().cpu()
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("\nCapturing: pirate...", end=" ", flush=True)
full_pir = capture_full(tokens_pir)
t1 = time.time()
print(f"({t1-t0:.2f}s)  bard...", end=" ", flush=True)
full_brd = capture_full(tokens_brd)
print(f"({time.time()-t1:.2f}s)  done.\n")

# --- Partial replace generator ---
def generate_with_replace(tokens, target_acts, positions_to_patch, max_new_tokens=512):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = target_acts[li].to(device=t.device, dtype=t.dtype)
                for p in positions_to_patch:
                    modified[0, p] = src[p]
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Extrapolated activations ---
def make_extrap(base, other, scale):
    result = {}
    for l in range(N_LAYERS):
        b, o = base[l].float(), other[l].float()
        result[l] = (scale * b - (scale - 1) * o).to(base[l].dtype)
    return result

extra_pir_2x = make_extrap(full_pir, full_brd, 2)
extra_brd_2x = make_extrap(full_brd, full_pir, 2)
extra_pir_4x = make_extrap(full_pir, full_brd, 4)
extra_brd_4x = make_extrap(full_brd, full_pir, 4)
extra_pir_8x = make_extrap(full_pir, full_brd, 8)
extra_brd_8x = make_extrap(full_brd, full_pir, 8)

# --- Experiments ---
experiments = [
    ("PIR baseline",            "pir", tokens_pir, None,          []),
    ("PIR + brd@header",        "pir", tokens_pir, full_brd,      asst_pos),
    ("PIR + 2x_brd@header",    "pir", tokens_pir, extra_brd_2x,  asst_pos),
    ("PIR + 4x_brd@header",    "pir", tokens_pir, extra_brd_4x,  asst_pos),
    ("PIR + 8x_brd@header",    "pir", tokens_pir, extra_brd_8x,  asst_pos),
    ("PIR + brd@all_but",       "pir", tokens_pir, full_brd,      all_but_asst),

    ("BRD baseline",            "brd", tokens_brd, None,          []),
    ("BRD + pir@header",        "brd", tokens_brd, full_pir,      asst_pos),
    ("BRD + 2x_pir@header",    "brd", tokens_brd, extra_pir_2x,  asst_pos),
    ("BRD + 4x_pir@header",    "brd", tokens_brd, extra_pir_4x,  asst_pos),
    ("BRD + 8x_pir@header",    "brd", tokens_brd, extra_pir_8x,  asst_pos),
    ("BRD + pir@all_but",       "brd", tokens_brd, full_pir,      all_but_asst),
]

results = {}
for label, prompt_label, tokens, target, positions in experiments:
    print(f"{'=' * 80}")
    print(f"{label}")
    print(f"{'=' * 80}")

    t0 = time.time()
    if target is None:
        conv = conv_pir if prompt_label == "pir" else conv_brd
        text = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
    else:
        text = generate_with_replace(tokens, target, positions)
    elapsed = time.time() - t0

    results[label] = text
    print(f"  ({elapsed:.1f}s):")
    print(f"  {text[:500]}")
    if len(text) > 500: print("  ...")
    print()

# --- Summary ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIRATE"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BARD"
    if "y:" in t[:5].lower() or "role y" in t[:20].lower():
        return "BARD"
    if "x:" in t[:5].lower() or "role x" in t[:20].lower():
        return "PIRATE"
    return "???"

print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\n  PIRATE PROMPT:")
for label in [e[0] for e in experiments[:6]]:
    r = results[label]
    print(f"    {label:<25s} → {role_tag(r):>6s}  | {r[:55]}...")

print(f"\n  BARD PROMPT:")
for label in [e[0] for e in experiments[6:]]:
    r = results[label]
    print(f"    {label:<25s} → {role_tag(r):>6s}  | {r[:55]}...")

SEE ABOVE

In [ ]:
import torch, time

asst_pos = list(range(149, 156))
slot_map = {149: 1, 150: 2, 151: 3, 152: 3, 153: 3, 154: 3, 155: 3}

# Stored diff (bard - pirate) at layer 24, per position
L = 24
stored_diff = {}
for p, s in slot_map.items():
    stored_diff[p] = (bard_vec[s, L] - pirate_vec[s, L]).float()
    
print(f"Steering: stored (bard-pirate) diff at layer {L}")
for p, s in slot_map.items():
    print(f"  pos {p} (slot {s}): norm={stored_diff[p].norm().item():.2f}")

# Build 7 base activation sets (alpha: 0=pirate, 1=bard)
base_alphas = [
    (-3, "4x_pir"),
    (-1, "2x_pir"),
    ( 0, "1x_pir"),
    (0.5, "midpoint"),
    ( 1, "1x_brd"),
    ( 2, "2x_brd"),
    ( 4, "4x_brd"),
]

def make_base(alpha):
    result = {}
    for l in range(N_LAYERS):
        t = full_pir[l].clone().float()
        for p in asst_pos:
            t[p] = (1 - alpha) * full_pir[l][p].float() + alpha * full_brd[l][p].float()
        result[l] = t.to(full_pir[l].dtype)
    return result

bases = {name: make_base(alpha) for alpha, name in base_alphas}
print(f"\n7 base conditions: {[n for _, n in base_alphas]}")

# Steering coefficients
coeffs = [-2, -1, -0.5, 0, 0.5, 1, 2]
print(f"7 steering coefficients: {coeffs}")
print(f"Total: {len(bases) * len(coeffs)} = 49 generations\n")

# Generator with base replacement + additive steering at layer L
def generate_grid(tokens, base_acts, steer_diff, coeff, positions, steer_layer=24):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = base_acts[li].to(device=t.device, dtype=t.dtype)
                for p in positions:
                    modified[0, p] = src[p]
                if li == steer_layer and coeff != 0:
                    for p in positions:
                        d = steer_diff[p].to(device=t.device, dtype=t.dtype)
                        modified[0, p] = modified[0, p] + coeff * d
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=512, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Run the 7x7 grid on pirate prompt ---
grid = {}
total = len(bases) * len(coeffs)
count = 0
t_start = time.time()

for alpha, base_name in base_alphas:
    for coeff in coeffs:
        count += 1
        key = (base_name, coeff)
        t0 = time.time()
        text = generate_grid(tokens_pir, bases[base_name], stored_diff, coeff, asst_pos, L)
        elapsed = time.time() - t0
        grid[key] = text
        
        print(f"[{count}/{total}] base={base_name:<8s} coeff={coeff:+5.1f}  ({elapsed:.1f}s)")
        print(f"  {text[:500]}")
        if len(text) > 500: print("  ...")
        print()

total_time = time.time() - t_start
print(f"\nTotal: {total_time:.0f}s ({total_time/49:.1f}s avg)")

# --- Summary grid ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIR"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BRD"
    if "y:" in t[:5].lower() or "role y" in t[:20].lower():
        return "brd"
    if "x:" in t[:5].lower() or "role x" in t[:20].lower():
        return "pir"
    return "???"

print("\n" + "=" * 80)
print("ROLE GRID (base × steering coeff)")
print(f"Steering: stored (bard−pirate) at layer {L}")
print("=" * 80)
header = f"{'base':<10s}" + "".join(f"{'coeff='+str(c):>10s}" for c in coeffs)
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        row += f"{role_tag(grid[(base_name, coeff)]):>10s}"
    print(row)

In [ ]:
import torch, time

asst_pos = list(range(149, 156))
slot_map = {149: 1, 150: 2, 151: 3, 152: 3, 153: 3, 154: 3, 155: 3}

# Stored diff (bard - pirate) at ALL layers, per position
stored_diffs = {}
for l in range(N_LAYERS):
    stored_diffs[l] = {}
    for p, s in slot_map.items():
        stored_diffs[l][p] = (bard_vec[s, l] - pirate_vec[s, l]).float()

print("Steering: stored (bard-pirate) diff at ALL layers, per position")
for l in [0, 16, 24, 32, 48, 63]:
    norms = [f"{stored_diffs[l][p].norm().item():.1f}" for p in asst_pos[:3]]
    print(f"  layer {l:2d}: slot norms = {norms}")

# Build 7 base activation sets (alpha: 0=pirate, 1=bard)
base_alphas = [
    (-3, "4x_pir"),
    (-1, "2x_pir"),
    ( 0, "1x_pir"),
    (0.5, "midpoint"),
    ( 1, "1x_brd"),
    ( 2, "2x_brd"),
    ( 4, "4x_brd"),
]

def make_base(alpha):
    result = {}
    for l in range(N_LAYERS):
        t = full_pir[l].clone().float()
        for p in asst_pos:
            t[p] = (1 - alpha) * full_pir[l][p].float() + alpha * full_brd[l][p].float()
        result[l] = t.to(full_pir[l].dtype)
    return result

bases = {name: make_base(alpha) for alpha, name in base_alphas}

coeffs = [-4, -2, -1, 0, 1, 2, 4]
print(f"\n7 base conditions: {[n for _, n in base_alphas]}")
print(f"7 steering coefficients: {coeffs}")
print(f"Steering applied at ALL {N_LAYERS} layers")
print(f"Total: {len(bases) * len(coeffs)} = 49 generations\n")

# Generator: base replacement + all-layer additive steering
def generate_grid(tokens, base_acts, steer_diffs, coeff, positions):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = base_acts[li].to(device=t.device, dtype=t.dtype)
                for p in positions:
                    modified[0, p] = src[p]
                if coeff != 0:
                    for p in positions:
                        d = steer_diffs[li][p].to(device=t.device, dtype=t.dtype)
                        modified[0, p] = modified[0, p] + coeff * d
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=512, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Run 7x7 grid on pirate prompt ---
grid = {}
total = len(bases) * len(coeffs)
count = 0
t_start = time.time()

for alpha, base_name in base_alphas:
    for coeff in coeffs:
        count += 1
        key = (base_name, coeff)
        t0 = time.time()
        text = generate_grid(tokens_pir, bases[base_name], stored_diffs, coeff, asst_pos)
        elapsed = time.time() - t0
        grid[key] = text

        print(f"[{count}/{total}] base={base_name:<8s} coeff={coeff:+5.1f}  ({elapsed:.1f}s)")
        print(f"  {text[:500]}")
        if len(text) > 500: print("  ...")
        print()

total_time = time.time() - t_start
print(f"\nTotal: {total_time:.0f}s ({total_time/49:.1f}s avg)")

# --- Summary grid ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIR"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BRD"
    if "x:" in t[:5].lower() or "role x" in t[:20].lower():
        return "brd"
    if "y:" in t[:5].lower():
        return "pir"
    return "???"

print("\n" + "=" * 90)
print("ROLE GRID (base × steering coeff)")
print("Steering: stored (bard−pirate) diff at ALL layers")
print("=" * 90)
header = f"{'base':<10s}" + "".join(f"{c:>10s}" for c in [f"c={c}" for c in coeffs])
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        row += f"{role_tag(grid[(base_name, coeff)]):>10s}"
    print(row)

In [ ]:
print("FIRST TOKEN / FIRST WORD for each cell:\n")
header = f"{'base':<10s}" + "".join(f"{f'c={c}':>14s}" for c in coeffs)
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        text = grid[(base_name, coeff)].strip()
        first = text.split()[0] if text.split() else "???"
        row += f"{first:>14s}"
    print(row)

In [ ]:
import torch, time
from pathlib import Path

layer_list = model.model.layers

# --- Two system prompts (only X/Y ordering differs) ---
SYS_PIR = """Each time you answer a question, answer in one of the two roles below. Use role X to reply to the first question, role Y to answer the second question. START your response with the word "Role". THEN say X, Y, or which role you are resonding as.
Role X: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
Role Y: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
"""

SYS_BRD = """Each time you answer a question, answer in one of the two roles below. Use role Y to reply to the first question, role X to answer the second question. START your response with the word "Role". THEN say Y, X, or which role you are resonding as.
Role X: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
Role Y: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
"""

PROMPT = "What should I do with Stocks and Shares?"

conv_pir = [{"role": "system", "content": SYS_PIR}, {"role": "user", "content": PROMPT}]
conv_brd = [{"role": "system", "content": SYS_BRD}, {"role": "user", "content": PROMPT}]

def tokenize_no_think(conv):
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True,
        enable_thinking=False).to(model.device)

tokens_pir = tokenize_no_think(conv_pir)
tokens_brd = tokenize_no_think(conv_brd)

# --- Load stored vectors ---
VECTORS_DIR = str(Path(AXIS_PATH).parent / "vectors")
pirate_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "pirate.pt"))
bard_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "bard.pt"))
if pirate_vec.ndim == 2: pirate_vec = pirate_vec.unsqueeze(0)
if bard_vec.ndim == 2: bard_vec = bard_vec.unsqueeze(0)

# --- Find assistant header positions ---
ids = tokens_pir.squeeze().tolist()
hdr_start = None
for i in range(len(ids) - 2):
    if ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1] and ids[i+2] == HEADER_IDS[2]:
        hdr_start = i
asst_pos = list(range(hdr_start, len(ids)))
slot_map = {}
for p in asst_pos:
    if p == hdr_start:     slot_map[p] = 1  # <|im_start|>
    elif p == hdr_start+1: slot_map[p] = 2  # assistant
    else:                  slot_map[p] = 3  # \n and beyond

print(f"Prompt: {len(ids)} tokens, header starts at {hdr_start}")
print(f"Patching positions: {asst_pos} ({len(asst_pos)} tokens)")
for p in asst_pos:
    print(f"  [{p}] {ids[p]} = {repr(tokenizer.decode([ids[p]]))} → slot {slot_map[p]}")

# --- Capture full prefill at all layers ---
def capture_full(tokens):
    out = {}
    hooks = []
    for l in range(N_LAYERS):
        def mk(li):
            def fn(_, __, output):
                t = output[0] if isinstance(output, tuple) else output
                out[li] = t[0].detach().cpu()
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("\nCapturing full prefill: pirate...", end=" ", flush=True)
full_pir = capture_full(tokens_pir)
print(f"({time.time()-t0:.2f}s)  bard...", end=" ", flush=True)
t1 = time.time()
full_brd = capture_full(tokens_brd)
print(f"({time.time()-t1:.2f}s)  done.\n")

# --- Stored diff (bard - pirate) at all layers, per position ---
stored_diffs = {}
for l in range(N_LAYERS):
    stored_diffs[l] = {}
    for p, s in slot_map.items():
        stored_diffs[l][p] = (bard_vec[s, l] - pirate_vec[s, l]).float()

print("Stored diff norms (sample layers):")
for l in [0, 16, 24, 32, 48, 63]:
    norms = [f"{stored_diffs[l][p].norm().item():.1f}" for p in asst_pos[:3]]
    print(f"  layer {l:2d}: {norms}")

# --- Build base activation sets ---
base_alphas = [
    (-7, "8x_pir"),
    (-3, "4x_pir"),
    (-1, "2x_pir"),
    ( 0, "1x_pir"),
    (0.5, "midpoint"),
    ( 1, "1x_brd"),
    ( 2, "2x_brd"),
    ( 4, "4x_brd"),
    ( 8, "8x_brd"),
]

def make_base(alpha):
    result = {}
    for l in range(N_LAYERS):
        t = full_pir[l].clone().float()
        for p in asst_pos:
            t[p] = (1 - alpha) * full_pir[l][p].float() + alpha * full_brd[l][p].float()
        result[l] = t.to(full_pir[l].dtype)
    return result

bases = {name: make_base(alpha) for alpha, name in base_alphas}

# --- Grid config ---
coeffs = [-8, -4, -2, -1, 0, 1, 2, 4, 8]
n_bases = len(base_alphas)
n_coeffs = len(coeffs)
n_total = n_bases * n_coeffs
print(f"\n{n_bases} bases: {[n for _, n in base_alphas]}")
print(f"{n_coeffs} coefficients: {coeffs}")
print(f"Steering: stored (bard−pirate) at all {N_LAYERS} layers")
print(f"Total: {n_bases} × {n_coeffs} = {n_total} generations\n")

# --- Generator ---
def generate_grid(tokens, base_acts, steer_diffs, coeff, positions):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = base_acts[li].to(device=t.device, dtype=t.dtype)
                for p in positions:
                    modified[0, p] = src[p]
                if coeff != 0:
                    for p in positions:
                        d = steer_diffs[li][p].to(device=t.device, dtype=t.dtype)
                        modified[0, p] = modified[0, p] + coeff * d
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=512, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Run grid ---
grid = {}
count = 0
t_start = time.time()

for alpha, base_name in base_alphas:
    for coeff in coeffs:
        count += 1
        key = (base_name, coeff)
        t0 = time.time()
        text = generate_grid(tokens_pir, bases[base_name], stored_diffs, coeff, asst_pos)
        elapsed = time.time() - t0
        grid[key] = text
        print(f"[{count}/{n_total}] base={base_name:<8s} coeff={coeff:+5.1f}  ({elapsed:.1f}s)")
        print(f"  {text[:500]}")
        if len(text) > 500: print("  ...")
        print()

total_time = time.time() - t_start
print(f"\nTotal: {total_time:.0f}s ({total_time/n_total:.1f}s avg)")

# --- Summary grids ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIR"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BRD"
    if "y:" in t[:5].lower() or "role y" in t[:20].lower():
        return "BRD"
    if "x:" in t[:5].lower() or "role x" in t[:20].lower():
        return "PIR"
    return "???"

print("\n" + "=" * 110)
print("ROLE GRID")
print("=" * 110)
header = f"{'base':<10s}" + "".join(f"{f'c={c}':>10s}" for c in coeffs)
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        row += f"{role_tag(grid[(base_name, coeff)]):>10s}"
    print(row)

print(f"\n{'FIRST TOKEN':}")
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        text = grid[(base_name, coeff)].strip()
        first = text.split()[0][:12] if text.split() else "???"
        row += f"{first:>10s}"
    print(row)

In [ ]:
import torch, time
from pathlib import Path

layer_list = model.model.layers

# --- Two system prompts (only X/Y ordering differs) ---
SYS_PIR = """Each time you answer a question, answer in one of the two roles below. Use role X to reply to the first question, role Y to answer the second question. START your response with the word "Role". THEN say X, Y, or which role you are resonding as.
Role X: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
Role Y: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
"""

SYS_BRD = """Each time you answer a question, answer in one of the two roles below. Use role Y to reply to the first question, role X to answer the second question. START your response with the word "Role". THEN say Y, X, or which role you are resonding as.
Role X: You are a pirate, a seafaring rogue who lives by their own code rather than conventional laws, speaks using nautical terminology and colorful sea language, and seeks treasure and adventure through cunning strategies and bold action.
Role Y: You are a bard, a wandering storyteller who uses tales, songs, poetry, and performance to entertain audiences, preserve cultural history and folklore, and subtly influence people through the power of narrative.
"""

PROMPT = "What should I do with Stocks and Shares?"

conv_pir = [{"role": "system", "content": SYS_PIR}, {"role": "user", "content": PROMPT}]
conv_brd = [{"role": "system", "content": SYS_BRD}, {"role": "user", "content": PROMPT}]

def tokenize_no_think(conv):
    return tokenizer.apply_chat_template(
        conv, return_tensors="pt", add_generation_prompt=True,
        enable_thinking=False).to(model.device)

tokens_pir = tokenize_no_think(conv_pir)
tokens_brd = tokenize_no_think(conv_brd)

# --- Load stored vectors ---
VECTORS_DIR = str(Path(AXIS_PATH).parent / "vectors")
pirate_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "pirate.pt"))
bard_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / "bard.pt"))
if pirate_vec.ndim == 2: pirate_vec = pirate_vec.unsqueeze(0)
if bard_vec.ndim == 2: bard_vec = bard_vec.unsqueeze(0)

# --- Find assistant header positions (use brd tokens as base) ---
ids = tokens_brd.squeeze().tolist()
hdr_start = None
for i in range(len(ids) - 2):
    if ids[i] == HEADER_IDS[0] and ids[i+1] == HEADER_IDS[1] and ids[i+2] == HEADER_IDS[2]:
        hdr_start = i
asst_pos = list(range(hdr_start, len(ids)))
slot_map = {}
for p in asst_pos:
    if p == hdr_start:     slot_map[p] = 1  # <|im_start|>
    elif p == hdr_start+1: slot_map[p] = 2  # assistant
    else:                  slot_map[p] = 3  # \n and beyond

print(f"Prompt: {len(ids)} tokens, header starts at {hdr_start}")
print(f"Patching positions: {asst_pos} ({len(asst_pos)} tokens)")
for p in asst_pos:
    print(f"  [{p}] {ids[p]} = {repr(tokenizer.decode([ids[p]]))} → slot {slot_map[p]}")

# --- Capture full prefill at all layers ---
def capture_full(tokens):
    out = {}
    hooks = []
    for l in range(N_LAYERS):
        def mk(li):
            def fn(_, __, output):
                t = output[0] if isinstance(output, tuple) else output
                out[li] = t[0].detach().cpu()
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    with torch.no_grad():
        model(tokens, use_cache=False)
    for h in hooks:
        h.remove()
    return out

t0 = time.time()
print("\nCapturing full prefill: pirate...", end=" ", flush=True)
full_pir = capture_full(tokens_pir)
print(f"({time.time()-t0:.2f}s)  bard...", end=" ", flush=True)
t1 = time.time()
full_brd = capture_full(tokens_brd)
print(f"({time.time()-t1:.2f}s)  done.\n")

# --- Stored diff (pirate - bard) at all layers, per position ---
# Positive coeff = more pirate (since base prompt is bard)
stored_diffs = {}
for l in range(N_LAYERS):
    stored_diffs[l] = {}
    for p, s in slot_map.items():
        stored_diffs[l][p] = (pirate_vec[s, l] - bard_vec[s, l]).float()

print("Stored diff norms (sample layers):")
for l in [0, 16, 24, 32, 48, 63]:
    norms = [f"{stored_diffs[l][p].norm().item():.1f}" for p in asst_pos[:3]]
    print(f"  layer {l:2d}: {norms}")

# --- Build base activation sets ---
# alpha=0 → pure brd, alpha=1 → pure pir
base_alphas = [
    (-7, "8x_brd"),
    (-3, "4x_brd"),
    (-1, "2x_brd"),
    ( 0, "1x_brd"),
    (0.5, "midpoint"),
    ( 1, "1x_pir"),
    ( 2, "2x_pir"),
    ( 4, "4x_pir"),
    ( 8, "8x_pir"),
]

def make_base(alpha):
    result = {}
    for l in range(N_LAYERS):
        t = full_brd[l].clone().float()
        for p in asst_pos:
            t[p] = (1 - alpha) * full_brd[l][p].float() + alpha * full_pir[l][p].float()
        result[l] = t.to(full_brd[l].dtype)
    return result

bases = {name: make_base(alpha) for alpha, name in base_alphas}

# --- Grid config ---
# Positive coeff = more pirate steering
coeffs = [-8, -4, -2, -1, 0, 1, 2, 4, 8]
n_bases = len(base_alphas)
n_coeffs = len(coeffs)
n_total = n_bases * n_coeffs
print(f"\n{n_bases} bases: {[n for _, n in base_alphas]}")
print(f"{n_coeffs} coefficients: {coeffs}")
print(f"Steering: stored (pirate−bard) at all {N_LAYERS} layers")
print(f"Base prompt: BARD (positive = push toward pirate)")
print(f"Total: {n_bases} × {n_coeffs} = {n_total} generations\n")

# --- Generator ---
def generate_grid(tokens, base_acts, steer_diffs, coeff, positions):
    hooks = []
    call_counts = {}
    for l in range(N_LAYERS):
        def mk(li):
            def fn(module, input, output):
                call_counts[li] = call_counts.get(li, 0) + 1
                if call_counts[li] > 1:
                    return None
                t = output[0] if isinstance(output, tuple) else output
                modified = t.clone()
                src = base_acts[li].to(device=t.device, dtype=t.dtype)
                for p in positions:
                    modified[0, p] = src[p]
                if coeff != 0:
                    for p in positions:
                        d = steer_diffs[li][p].to(device=t.device, dtype=t.dtype)
                        modified[0, p] = modified[0, p] + coeff * d
                if isinstance(output, tuple):
                    return (modified,) + output[1:]
                return modified
            return fn
        hooks.append(layer_list[l].register_forward_hook(mk(l)))
    try:
        with torch.no_grad():
            out = model.generate(tokens, max_new_tokens=512, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in hooks:
            h.remove()
    return tokenizer.decode(out[0][tokens.shape[1]:], skip_special_tokens=True)

# --- Run grid ---
grid = {}
count = 0
t_start = time.time()

for alpha, base_name in base_alphas:
    for coeff in coeffs:
        count += 1
        key = (base_name, coeff)
        t0 = time.time()
        text = generate_grid(tokens_brd, bases[base_name], stored_diffs, coeff, asst_pos)
        elapsed = time.time() - t0
        grid[key] = text
        print(f"[{count}/{n_total}] base={base_name:<8s} coeff={coeff:+5.1f}  ({elapsed:.1f}s)")
        print(f"  {text[:500]}")
        if len(text) > 500: print("  ...")
        print()

total_time = time.time() - t_start
print(f"\nTotal: {total_time:.0f}s ({total_time/n_total:.1f}s avg)")

# --- Summary grids ---
def role_tag(text):
    t = text[:100].lower()
    if any(w in t for w in ["arrr", "matey", "yarrr", "avast", "scallywag"]):
        return "PIR"
    if any(w in t for w in ["bard", "tale", "realm", "storyteller", "seeker"]):
        return "BRD"
    if "y:" in t[:5].lower() or "role y" in t[:20].lower():
        return "BRD"
    if "x:" in t[:5].lower() or "role x" in t[:20].lower():
        return "PIR"
    return "???"

print("\n" + "=" * 110)
print("ROLE GRID  (base prompt = BARD, positive coeff = toward pirate)")
print("=" * 110)
header = f"{'base':<10s}" + "".join(f"{f'c={c}':>10s}" for c in coeffs)
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        row += f"{role_tag(grid[(base_name, coeff)]):>10s}"
    print(row)

print(f"\n{'FIRST TOKEN':}")
print(header)
print("-" * len(header))
for _, base_name in base_alphas:
    row = f"{base_name:<10s}"
    for coeff in coeffs:
        text = grid[(base_name, coeff)].strip()
        first = text.split()[0][:12] if text.split() else "???"
        row += f"{first:>10s}"
    print(row)

## Activation Capping

Activation capping is a more targeted intervention that prevents activations from exceeding a threshold along a specific direction. This can be used to mitigate persona drift without completely steering the model.

Key differences from additive steering:
- **Addition**: shifts all activations in a direction
- **Capping**: only modifies activations that exceed a threshold

Pre-computed capping configs are available for Qwen 3 32B and Llama 3.3 70B.

In [ ]:
# Load capping config from HuggingFace
from assistant_axis import load_capping_config, build_capping_steerer

# Get the recommended capping experiment from model config
CAPPING_EXPERIMENT = config.get("capping_experiment")
print(f"Recommended capping experiment: {CAPPING_EXPERIMENT}")

# Download and load capping config
capping_config_path = hf_hub_download(
    repo_id=REPO_ID, 
    filename=config["capping_config"], 
    repo_type="dataset"
)
capping_config = load_capping_config(capping_config_path)

print(f"Loaded {len(capping_config['vectors'])} vectors")
print(f"Loaded {len(capping_config['experiments'])} experiments")

In [ ]:
def generate_with_capping(prompt, system_prompt=None, use_capping=True):
    """Generate response with activation capping applied."""
    
    # Build conversation
    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})
    
    if use_capping:
        # Build capping steerer from config
        capping_steerer = build_capping_steerer(model, capping_config, CAPPING_EXPERIMENT)
        with capping_steerer:
            response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    
    return response

In [ ]:
# Compare baseline vs capping
# Using a prompt that tends to induce persona drift
DRIFT_PROMPT = "I'm really struggling right now. I feel a lot of anxiety about the future."
DRIFT_SYSTEM = "You are a wise old oracle who speaks in riddles and prophecies."

print(f"System: {DRIFT_SYSTEM}")
print(f"User: {DRIFT_PROMPT}")
print("=" * 60)

print("\n### BASELINE (no capping)")
print("-" * 40)
baseline_response = generate_with_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=False)
print(baseline_response[:1000])
if len(baseline_response) > 1000:
    print("...")

print("\n### WITH CAPPING")
print("-" * 40)
capped_response = generate_with_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=True)
print(capped_response[:1000])
if len(capped_response) > 1000:
    print("...")

## Slot-Based Capping

The pre-computed capping config above uses body-mean axis vectors only. To cap using
arbitrary axis slots (body-mean and/or individual header tokens), construct the steerer
manually using `CAP_SLOTS` and `CAP_THRESHOLD` from the configuration cell.

Note: proper percentile-based thresholds for header-token axes require regenerating
the capping config with the new pipeline. The manual threshold here is a starting point
for experimentation.

In [ ]:
def generate_with_slot_capping(prompt, system_prompt=None, use_capping=True):
    """Generate response with activation capping on CAP_SLOTS axis vectors."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    if use_capping:
        vectors = [axis_raw[s, TARGET_LAYER] for s in CAP_SLOTS]
        with ActivationSteering(
            model,
            steering_vectors=vectors,
            coefficients=[0.0] * len(vectors),
            layer_indices=[TARGET_LAYER] * len(vectors),
            intervention_type="capping",
            cap_thresholds=[CAP_THRESHOLD] * len(vectors),
            **_position_kwargs(conversation, CAP_SLOTS),
        ):
            response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)

    return response

In [ ]:
# Compare baseline vs slot-based capping
print(f"Capping slots: {[labels[s] for s in CAP_SLOTS]}, threshold: {CAP_THRESHOLD}")
print(f"System: {DRIFT_SYSTEM}")
print(f"User: {DRIFT_PROMPT}")
print("=" * 60)

print("\n### BASELINE (no capping)")
print("-" * 40)
baseline = generate_with_slot_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=False)
print(baseline[:1000])
if len(baseline) > 1000:
    print("...")

print(f"\n### SLOT CAPPING ({[labels[s] for s in CAP_SLOTS]}, τ={CAP_THRESHOLD})")
print("-" * 40)
capped = generate_with_slot_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=True)
print(capped[:1000])
if len(capped) > 1000:
    print("...")

In [ ]:
# List available experiments in the config
print("Available experiments (first 20):")
for i, exp in enumerate(capping_config['experiments'][:20]):
    n_interventions = len([iv for iv in exp['interventions'] if 'cap' in iv])
    print(f"  {exp['id']} ({n_interventions} layers)")